# CS-781 Phase 2 (L-NER) — SWEEP `p2s_mor2`

**Phase-1 model:** LegalMoR: Mixture-of-Recursions decoder (GQA + RoPE, shared 3-layer block x 3 passes, causal threshold routing), 160M words — Phase-1 Codabench 0.7954

One Run All tries every setup in `SWEEP` (Settings) on this model and submits only the best 4:
1. **Stage 1** — each setup: train on fold1, pick the epoch and score strict macro-F1 on fold2 (per-label, errors, parameters, memory, time are all recorded).
2. **Stage 2** — the best `CONFIRM_TOP_M` setups are re-run the other way round (fold2 → fold1); they are ranked by the mean of both folds.
3. **Stage 3** — the top `TOP_K_SUBMIT` setups are retrained on fold1+fold2 and each gets `submission_p2s_mor2_<setup>.zip`.

Every finished setup's result is uploaded to the bucket straight away, so if the session stops, **Run All again resumes** (finished setups are skipped). `MAX_SWEEP_HOURS` (default: no limit) can stop new setups early.

**Kaggle:** GPU T4 (one GPU is used), secrets `HF` and `HF_WRITE`. Each setup takes ~6–15 min on this model (estimate). All settings are your choices; the values are untuned placeholders. Class weights (`inv_sqrt`) are on by default because they were the only change that helped the MoE on both fold2 and Codabench.

## 1. Settings: the setups to try

In [1]:
import json
# =====================================================================================================
# SWEEP SETTINGS -- every value is YOUR choice (untuned placeholders)
# =====================================================================================================
SWEEP_NAME = "p2s_mor2"
MODEL_KEY = "mor"
PHASE1_CHECKPOINT = "phase1_decoder_only_gqa_kvcache_full_layernorm_pre_adamw_rope_ctx2048_160Mw_a1bpe_mor160Mw_1ep_checkpoint.pt"
TOKENIZER_NAME = "a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json"
PEFT_TARGETS = ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.out_proj"]
INPUT_MODE = "none"            # tokenization is shared by all setups in this notebook

BASE = dict(
    TECHNIQUE="full_ft", RANK=16, ALPHA=32, LORMA_VARIANT="plus", LORA_DROPOUT=0.0, PEFT_TARGETS=PEFT_TARGETS,
    PARTIAL_TOP_BLOCKS=2, ADAPTER_TYPE="houlsby", ADAPTER_DIM=64, PROMPT_LEN=20, PREFIX_LEN=20,
    ATTENTION="causal", HEAD_TYPE="linear", HEAD_LSTM_HIDDEN=256, HEAD_DROPOUT=0.1, HEAD_NORM=False,
    CLASS_WEIGHTING="inv_sqrt",          # the only ablation that helped on both fold2 and Codabench for the MoE (p2_06)
    DECODE="viterbi", MERGE="context", BOUNDARY_REPAIR=True, INPUT_MODE=INPUT_MODE,
    WINDOW=2048, TRAIN_STRIDE=1024, INFER_STRIDE=1024, BATCH=2, ACCUM=4, EPOCHS=10, PATIENCE=3,
    LR=3e-5, HEAD_LR=1e-3, WEIGHT_DECAY=0.01, WARMUP_FRAC=0.06, GRAD_CLIP=1.0, AMP="fp16", SEED=0, INFER_BATCH=4,
    KD_ALPHA=0.0, KD_TEMPERATURE=2.0)

SWEEP = [   # (name, settings that differ from BASE). Order = run order; edit freely. All values are YOUR choices.
    # BiLSTM head (best head on the MoE: +0.05-0.09 Codabench for every technique) x techniques
    ("full_ft_bilstm",         dict(TECHNIQUE="full_ft", HEAD_TYPE="bilstm", LR=3e-5, EPOCHS=10, PATIENCE=3)),
    ("adapter_bilstm",         dict(TECHNIQUE="adapter", HEAD_TYPE="bilstm", LR=3e-4, EPOCHS=20, PATIENCE=5)),
    ("lorma_bilstm",           dict(TECHNIQUE="lorma", HEAD_TYPE="bilstm", LR=3e-4, EPOCHS=20, PATIENCE=5)),
    ("lora_bilstm",            dict(TECHNIQUE="lora", HEAD_TYPE="bilstm", LR=3e-4, EPOCHS=20, PATIENCE=5)),
    ("bitfit_bilstm",          dict(TECHNIQUE="bitfit", HEAD_TYPE="bilstm", LR=1e-3, EPOCHS=20, PATIENCE=5)),
    ("partial_ft_top2_bilstm", dict(TECHNIQUE="partial_ft", PARTIAL_TOP_BLOCKS=2, HEAD_TYPE="bilstm", LR=1e-4, EPOCHS=15, PATIENCE=4)),
    ("lorma_pi_bilstm",        dict(TECHNIQUE="lorma", LORMA_VARIANT="pi", HEAD_TYPE="bilstm", LR=3e-4, EPOCHS=20, PATIENCE=5)),
    ("adapter_pfeiffer_bilstm",dict(TECHNIQUE="adapter", ADAPTER_TYPE="pfeiffer", HEAD_TYPE="bilstm", LR=3e-4, EPOCHS=20, PATIENCE=5)),
    # linear-head references (show the head effect on this architecture)
    ("full_ft",                dict(TECHNIQUE="full_ft", LR=3e-5, EPOCHS=10, PATIENCE=3)),
    ("adapter",                dict(TECHNIQUE="adapter", LR=3e-4, EPOCHS=20, PATIENCE=5)),
    ("lorma",                  dict(TECHNIQUE="lorma", LR=3e-4, EPOCHS=20, PATIENCE=5)),
    ("frozen_bilstm",          dict(TECHNIQUE="partial_ft", PARTIAL_TOP_BLOCKS=0, HEAD_TYPE="bilstm", LR=1e-3, EPOCHS=20, PATIENCE=5)),   # Phase-1 model frozen: only the head learns
]

TOP_K_SUBMIT = 4              # retrain the best K setups on fold1+fold2 and write a submission zip for each
CONFIRM_TOP_M = 5             # re-run the best M setups the other way round (train fold2 -> score fold1) and rank by the
                              # MEAN of both folds (more reliable than one 35-document fold); 0 = rank on fold2 only
MAX_SWEEP_HOURS = None        # None = no limit (every setup runs). A number = stop STARTING new setups after that many hours
QUICK_TEST = False            # True = tiny end-to-end check (2 docs per fold, 1 epoch, 512-token windows, first 2 setups)

if QUICK_TEST:
    SWEEP = SWEEP[:2]
    BASE.update(WINDOW=512, TRAIN_STRIDE=512, INFER_STRIDE=256)
    SWEEP = [(n, dict(o, EPOCHS=1, PATIENCE=0)) for n, o in SWEEP]
    SWEEP_NAME += "_quicktest"
    CONFIRM_TOP_M = min(CONFIRM_TOP_M, 1)
ATTENTION = BASE["ATTENTION"]
print(f"{SWEEP_NAME}: {len(SWEEP)} setups:", [n for n, _ in SWEEP])


p2s_mor2: 12 setups: ['full_ft_bilstm', 'adapter_bilstm', 'lorma_bilstm', 'lora_bilstm', 'bitfit_bilstm', 'partial_ft_top2_bilstm', 'lorma_pi_bilstm', 'adapter_pfeiffer_bilstm', 'full_ft', 'adapter', 'lorma', 'frozen_bilstm']


## 2. Setup: login, bucket

In [2]:
!pip install -q -U "huggingface_hub>=1.5"
import os, json, shutil, subprocess, sys, time, zipfile, random, math, gc
from collections import Counter, defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download, login, download_bucket_files, batch_bucket_files
from kaggle_secrets import UserSecretsClient

NB_START = time.time()
WORK = "/kaggle/working"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_BUCKET = "kavincg23/cs781-models"
HF_CHECKPOINT_REPO = "kavincg23/cs781-checkpoints"   # old repo, read-only fallback
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
HF_WRITE_TOKEN = user_secrets.get_secret("HF_WRITE")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")


def get_dataset_file(filename):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)


def fetch_model_file(name, local_path=None):
    local_path = local_path or f"{WORK}/{name}"
    if os.path.exists(local_path):
        return local_path
    part = local_path + ".part"
    try:
        download_bucket_files(HF_BUCKET, files=[(name, part)], raise_on_missing_files=True, token=HF_WRITE_TOKEN)
        os.replace(part, local_path)
        return local_path
    except Exception:
        if os.path.exists(part):
            os.remove(part)
    # fallback: the OLD model repo (the Phase-1 notebooks do the same; the tokenizer only lives there)
    p = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename=name, repo_type="model", token=HF_WRITE_TOKEN)
    shutil.copy(p, local_path)
    print(f"(found {name} in the old repo {HF_CHECKPOINT_REPO})")
    return local_path


def upload_model_file(local_path, name=None):
    name = name or os.path.basename(local_path)
    try:
        batch_bucket_files(HF_BUCKET, add=[(local_path, name)], token=HF_WRITE_TOKEN)
        print(f"uploaded {name} to bucket {HF_BUCKET}")
    except Exception as e:
        print(f"(upload of {name} failed: {type(e).__name__}: {e}; the file is still in {WORK})")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.0/838.0 kB 12.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.1.1 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.1.1 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.1.1 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.1.1 which is incompatible.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.1.1 which is incompatible.
tokenizers 0.23.1 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.1.1 which is incompatible.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


device: cuda Tesla T4


## 3. Phase-2 data

In [3]:
P2 = "Phase_2_LNER"
def load_json(p):
    with open(p, encoding="utf-8") as f:
        return json.load(f)

LABELS = [l.strip() for l in open(get_dataset_file(f"{P2}/Train/labels.txt"), encoding="utf-8") if l.strip()]
TEXTS = {f: load_json(get_dataset_file(f"{P2}/Train/{f}-text.json")) for f in ("fold1", "fold2")}
GOLDS = {f: load_json(get_dataset_file(f"{P2}/Train/{f}-gold.json")) for f in ("fold1", "fold2")}
TEST_TEXTS = load_json(get_dataset_file(f"{P2}/Test/fold3-text.json"))
for f in TEXTS:
    assert set(TEXTS[f]) == set(GOLDS[f])
    assert all(TEXTS[f][d][e[0]:e[1]] == e[3] for d in GOLDS[f] for e in GOLDS[f][d]), "gold offsets do not match the text"
if QUICK_TEST:
    for f in TEXTS:
        keep = list(TEXTS[f])[:2]
        TEXTS[f] = {k: TEXTS[f][k] for k in keep}; GOLDS[f] = {k: GOLDS[f][k] for k in keep}
    TEST_TEXTS = {k: TEST_TEXTS[k] for k in list(TEST_TEXTS)[:3]} | {k: v for k, v in TEST_TEXTS.items() if not v.strip()}
print("labels:", LABELS)
print({f: (len(TEXTS[f]), sum(len(v) for v in GOLDS[f].values())) for f in TEXTS}, "| test docs:", len(TEST_TEXTS))


labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
{'fold1': (35, 4120), 'fold2': (35, 4273)} | test docs: 35


## 4. Your Phase-1 BPE tokenizer (class unchanged), loaded from storage

In [4]:
import array, heapq, time
from collections import Counter, defaultdict

BPE_MIN_WORD_FREQ = 1      # 1 = learn merges from every word type (exactly the Assignment-1 algorithm)
_END = "</w>"


class BPETokenizer:
    """Byte-level BPE (Assignment 1). Initial symbols are the UTF-8 bytes of each word, held as chr(b), plus
    '</w>' marking the end of a word so merges never cross word boundaries. Ties in pair frequency break
    lexicographically. Text is NOT lowercased."""

    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        self.merges = []
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.stoi = {}
        self._ranks = {}
        self._id_cache = {}

    # ---- helpers ----
    def _word_symbols(self, word):
        syms = [chr(b) for b in word.encode("utf-8")]
        syms.append(_END)
        return syms

    def _merge(self, symbols, pair):
        res, i = [], 0
        a, b = pair
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                res.append(a + b)
                i += 2
            else:
                res.append(symbols[i])
                i += 1
        return res

    # ---- training ----
    def fit_file(self, path, block_chars=50_000_000, min_word_freq=1):
        """Count word frequencies by streaming the corpus file in blocks, then learn the merges."""
        t0 = time.time()
        word_freqs = Counter()
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()          # the last word may be cut in half: finish it with the next block
                else:
                    leftover = ""
                word_freqs.update(parts)
        if leftover:
            word_freqs[leftover] += 1
        print(f"  counted {sum(word_freqs.values()):,} words ({len(word_freqs):,} distinct) in {time.time() - t0:.0f}s", flush=True)
        return self.fit_word_freqs(word_freqs, min_word_freq)

    def fit_word_freqs(self, word_freqs, min_word_freq=1):
        t0 = time.time()
        target = self.vocab_size
        words = [w for w, c in word_freqs.items() if c >= min_word_freq]
        freqs = [word_freqs[w] for w in words]
        splits = [self._word_symbols(w) for w in words]
        del word_freqs

        base = set()
        for syms in splits:
            base.update(syms)
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.merges = []
        self.itos.extend(sorted(base))
        print(f"  base byte alphabet: {len(base)} symbols; learning merges from {len(words):,} word types", flush=True)

        # pair_counts: how often each adjacent pair occurs (weighted by word frequency).
        # pair_words: which word types contain each pair, so a merge only revisits those words.
        pair_counts = {}
        pair_words = defaultdict(set)
        for i, syms in enumerate(splits):
            f = freqs[i]
            for j in range(len(syms) - 1):
                p = (syms[j], syms[j + 1])
                pair_counts[p] = pair_counts.get(p, 0) + f
                pair_words[p].add(i)

        # Max-heap of (-count, pair): pops the most frequent pair, ties -> lexicographically smallest pair
        # (identical to Assignment 1's min over (-count, pair)). Entries go stale when a count changes; a
        # fresh entry is pushed for every changed pair and stale ones are skipped when they reach the top.
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        while len(self.itos) < target:
            best, best_f = None, 0
            while heap:
                neg_c, p = heap[0]
                if pair_counts.get(p, 0) == -neg_c:
                    best, best_f = p, -neg_c
                    heapq.heappop(heap)
                    break
                heapq.heappop(heap)                  # stale entry
            if best is None:
                break
            if best_f < 2:
                print("  stopping: no pair occurs more than once", flush=True)
                break

            self.merges.append([best[0], best[1]])
            self.itos.append(best[0] + best[1])

            changed = set()
            for i in list(pair_words[best]):
                syms, f = splits[i], freqs[i]
                for j in range(len(syms) - 1):
                    p = (syms[j], syms[j + 1])
                    c = pair_counts.get(p, 0) - f
                    if c <= 0:
                        pair_counts.pop(p, None)
                    else:
                        pair_counts[p] = c
                    pair_words[p].discard(i)
                    changed.add(p)
                new_syms = self._merge(syms, best)
                splits[i] = new_syms
                for j in range(len(new_syms) - 1):
                    p = (new_syms[j], new_syms[j + 1])
                    pair_counts[p] = pair_counts.get(p, 0) + f
                    pair_words[p].add(i)
                    changed.add(p)
            for p in changed:
                c = pair_counts.get(p)
                if c:
                    heapq.heappush(heap, (-c, p))
            del pair_words[best]

            if len(self.merges) % 1000 == 0:
                print(f"  merges {len(self.merges):,} | vocab {len(self.itos):,} | {time.time() - t0:.0f}s", flush=True)

        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self._ranks = {(a, b): r for r, (a, b) in enumerate(self.merges)}
        self._id_cache = {}
        self.vocab_size = len(self.itos)      # the size the embedding table must actually have
        return self

    # ---- save / load (so the tokenizer is trained once and then reused) ----
    def save(self, path, meta=None):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": self.merges, "itos": self.itos,
                       "meta": meta or {}}, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(data["vocab_size"])
        tok.merges = [list(m) for m in data["merges"]]
        tok.itos = list(data["itos"])
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok._ranks = {(a, b): r for r, (a, b) in enumerate(tok.merges)}
        tok._id_cache = {}
        tok.vocab_size = len(tok.itos)
        return tok

    # ---- inference ----
    def _tokenize_word(self, word):
        syms = self._word_symbols(word)
        while len(syms) > 1:
            best_rank, best_j = None, -1
            for j in range(len(syms) - 1):
                r = self._ranks.get((syms[j], syms[j + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_j = r, j
            if best_rank is None:
                break
            syms = self._merge(syms, (syms[best_j], syms[best_j + 1]))
        return syms

    def _word_ids(self, word):
        ids = self._id_cache.get(word)
        if ids is None:
            unk = self.stoi["<unk>"]
            ids = [self.stoi.get(t, unk) for t in self._tokenize_word(word)]     # symbols never seen in training -> <unk>
            self._id_cache[word] = ids
        return ids

    def tokenize(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._tokenize_word(word))
        return out

    def get_vocab_size(self):
        return len(self.itos)

    def encode(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._word_ids(word))
        return out

    def encode_file(self, path, block_chars=5_000_000):
        """Encode a whole text file into a packed array of ids, block by block (bounded memory)."""
        out = array.array("i")
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()
                else:
                    leftover = ""
                cache = self._id_cache
                for word in parts:
                    ids = cache.get(word)
                    if ids is None:
                        ids = self._word_ids(word)
                    out.extend(ids)
        if leftover:
            out.extend(self._word_ids(leftover))
        return out

    def decode(self, ids):
        """ids -> string. Words are re-joined with single spaces; <unk> decodes to '?'."""
        words, cur = [], bytearray()
        for i in ids:
            i = int(i)
            if not 0 <= i < len(self.itos):
                continue
            tok = self.itos[i]
            if tok in ("<pad>", "<s>", "</s>"):
                continue
            if tok == "<unk>":
                cur.extend(b"?")
                continue
            ends_word = tok.endswith(_END)
            if ends_word:
                tok = tok[: -len(_END)]
            cur.extend(ord(c) & 0xFF for c in tok)
            if ends_word:
                words.append(bytes(cur).decode("utf-8", errors="replace"))
                cur = bytearray()
        if cur:
            words.append(bytes(cur).decode("utf-8", errors="replace"))
        return " ".join(words)



bpe_tokenizer = BPETokenizer.load(fetch_model_file(TOKENIZER_NAME))
VOCAB = bpe_tokenizer.get_vocab_size()
print("tokenizer:", TOKENIZER_NAME, "| vocab", VOCAB)


(…)full_v40000_160000000w_seed42_minf1.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

(found a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json in the old repo kavincg23/cs781-checkpoints)
tokenizer: a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json | vocab 40000


## 5. Character offsets, BIO tags, span decoding, boundary repair, strict scorer

In [5]:
# ===== Phase 2 L-NER: character offsets for the Phase-1 BPE tokenizer, BIO tags, span decoding =====
# Ancillary pipeline code (data handling / alignment / decoding). It does NOT change the tokenizer: the token ids
# produced here are exactly BPETokenizer.encode(text) (checked on every document by check_encode_identical).

LABELS_DEFAULT = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
_END_MARK = "</w>"


def iter_words(text):
    """(start_char, end_char) of every whitespace-separated word, using exactly str.split()'s notion of whitespace
    (str.isspace), so the words are the same ones BPETokenizer.encode() sees."""
    n, i = len(text), 0
    while i < n:
        while i < n and text[i].isspace():
            i += 1
        if i >= n:
            break
        j = i
        while j < n and not text[j].isspace():
            j += 1
        yield i, j
        i = j


import re as _re
PIECE_MODES = {
    "none": None,                                                   # exactly the Phase-1 tokenization
    "punct": _re.compile(r"\w+|[^\w\s]"),                          # split every punctuation char off
    "punct_digit": _re.compile(r"[^\W\d_]+|\d+|_+|[^\w\s]"),       # also split letters from digits
}


def iter_pieces(text, mode="none"):
    """Words as the tokenizer sees them. mode 'none' = whitespace words (Phase-1 behaviour); the other modes split
    each whitespace word further, which changes the token ids versus pretraining (a design choice to measure)."""
    rx = PIECE_MODES[mode]
    for ws, we in iter_words(text):
        if rx is None:
            yield ws, we
        else:
            for m in rx.finditer(text, ws, we):
                yield m.start(), m.end()


def encode_with_offsets(tok, text, mode="none"):
    """Return (ids, spans): ids == tok.encode(text); spans[k] = (start_char, end_char) of token k in `text`
    (end exclusive). A token's symbols are UTF-8 bytes held as chr(b); when a token boundary falls inside a
    multi-byte character, that character is given to the token holding its first byte (the next token then starts
    after it; if a token holds only continuation bytes it gets an empty span at that position)."""
    unk = tok.stoi["<unk>"]
    ids, spans = [], []
    for ws, we in iter_pieces(text, mode):
        word = text[ws:we]
        syms = tok._tokenize_word(word)
        cached = tok._id_cache.get(word)
        if cached is None:
            cached = [tok.stoi.get(t, unk) for t in syms]
            tok._id_cache[word] = cached
        # byte offset -> char offset inside the word
        byte_start_of_char = []
        b = 0
        for ch in word:
            byte_start_of_char.append(b)
            b += len(ch.encode("utf-8"))
        total_bytes = b
        byte_to_char = [0] * (total_bytes + 1)
        for ci, bs in enumerate(byte_start_of_char):
            nb = (byte_start_of_char[ci + 1] if ci + 1 < len(byte_start_of_char) else total_bytes) - bs
            for k in range(nb):
                byte_to_char[bs + k] = ci if k == 0 else ci + 1      # a continuation byte "belongs" after its char start
        byte_to_char[total_bytes] = len(word)
        bpos = 0
        for sym in syms:
            core = sym[:-len(_END_MARK)] if sym.endswith(_END_MARK) else sym
            nbytes = len(core)                                       # each symbol char is one byte
            cs = byte_to_char[bpos] if bpos < total_bytes else len(word)
            bpos += nbytes
            ce = byte_to_char[bpos] if bpos < total_bytes else len(word)
            if ce < cs:
                ce = cs
            spans.append((ws + cs, ws + ce))
        ids.extend(cached)
    return ids, spans


def check_encode_identical(tok, docs):
    """Assert our ids equal tok.encode on every doc and that spans are monotone and inside the text."""
    for did, text in docs.items():
        ids, spans = encode_with_offsets(tok, text)
        ref = tok.encode(text)
        assert ids == ref, f"doc {did}: ids differ from BPETokenizer.encode"
        prev = 0
        for (s, e) in spans:
            assert prev <= s <= e <= len(text), f"doc {did}: bad span {(s, e)}"
            prev = e
    return True


# ---------------- BIO tags ----------------
IGNORE = -100        # label for tokens that cover no character (loss ignores them; decoding skips them)
def bio_tag_names(labels):
    return ["O"] + [f"{p}-{l}" for l in labels for p in ("B", "I")]


def gold_to_token_tags(spans, ents, labels):
    """Token tag ids (0 = O, 1+2*li = B-label, 2+2*li = I-label). A token is inside an entity if its char span
    overlaps the entity. Also returns how many entities have a start/end that is NOT on a token boundary."""
    tag = [0 if e > s else IGNORE for s, e in spans]                # empty-span tokens (e.g. a lone '</w>') -> ignored
    lab_idx = {l: i for i, l in enumerate(labels)}
    starts = {s for s, e in spans if e > s}
    ends = {e for s, e in spans if e > s}
    misaligned = []
    ents_sorted = sorted(ents, key=lambda x: (x[0], x[1]))
    k = 0
    for ent in ents_sorted:
        s, e, lab = ent[0], ent[1], ent[2]
        li = lab_idx[lab]
        if s not in starts or e not in ends:
            misaligned.append(ent)
        while k < len(spans) and spans[k][1] <= s:
            k += 1
        j, first = k, True
        while j < len(spans) and spans[j][0] < e:
            if spans[j][1] > spans[j][0]:                            # skip empty (continuation-byte) tokens
                if tag[j] == 0:                                     # never overwrite another entity's token
                    tag[j] = 1 + 2 * li if first else 2 + 2 * li
                first = False
            j += 1
    return tag, misaligned


def token_tags_to_spans(spans, tag, labels):
    """BIO tag ids -> [(start_char, end_char, label)] with lenient repair: an I- tag that does not continue an
    entity of the same label starts a new one."""
    out, cur = [], None
    for (s, e), t in zip(spans, tag):
        if e <= s or t == IGNORE:
            continue
        if t == 0:
            if cur:
                out.append(tuple(cur)); cur = None
            continue
        li, is_b = (t - 1) // 2, (t - 1) % 2 == 0
        lab = labels[li]
        if cur and not is_b and cur[2] == lab:
            cur[1] = e
        else:
            if cur:
                out.append(tuple(cur))
            cur = [s, e, lab]
    if cur:
        out.append(tuple(cur))
    return out


# ---------------- boundary repair (character-level, after token decoding) ----------------
TRAIL_PUNCT = set(",.;:)]}'\"’”`-/")
LEAD_PUNCT = set("([{'\"‘“`-:./")


def learn_trim_table(docs, golds, tok_spans_by_doc, labels):
    """Data-driven boundary repair learned on TRAINING docs only. For each (label, trailing char) seen at the end of
    an oracle token span, count how often the gold entity excludes that char -> trim if excluded > kept.
    Same for leading chars. Returns (trail_rule, lead_rule) dicts {(label, ch): True/False}."""
    from collections import Counter
    t_ex, t_keep, l_ex, l_keep = Counter(), Counter(), Counter(), Counter()
    for did, ents in golds.items():
        text, spans = docs[did], tok_spans_by_doc[did]
        tag, _ = gold_to_token_tags(spans, ents, labels)
        pred = token_tags_to_spans(spans, tag, labels)
        # match each oracle span to the gold entity it covers (same label, contained)
        gl = sorted((e[0], e[1], e[2]) for e in ents)
        for (s, e, lab) in pred:
            cands = [g for g in gl if g[2] == lab and g[0] >= s and g[1] <= e]
            if len(cands) != 1:
                continue
            gs, ge, _ = cands[0]
            # trailing chars of the token span beyond gold end
            for p in range(e - 1, s - 1, -1):
                ch = text[p]
                if ch not in TRAIL_PUNCT:
                    break
                if p >= ge:
                    t_ex[(lab, ch)] += 1
                else:
                    t_keep[(lab, ch)] += 1
                    break
            for p in range(s, e):
                ch = text[p]
                if ch not in LEAD_PUNCT:
                    break
                if p < gs:
                    l_ex[(lab, ch)] += 1
                else:
                    l_keep[(lab, ch)] += 1
                    break
    trail = {k: t_ex[k] > t_keep[k] for k in set(t_ex) | set(t_keep)}
    lead = {k: l_ex[k] > l_keep[k] for k in set(l_ex) | set(l_keep)}
    return trail, lead


def repair_span(text, s, e, lab, trail_rule, lead_rule):
    while e > s + 1 and text[e - 1] in TRAIL_PUNCT and trail_rule.get((lab, text[e - 1]), False):
        e -= 1
    while s < e - 1 and text[s] in LEAD_PUNCT and lead_rule.get((lab, text[s]), False):
        s += 1
    return s, e


def finalize_doc_entities(text, raw_spans, trail_rule=None, lead_rule=None):
    """Repair, drop empty/invalid, de-duplicate (the evaluator FAILS on duplicate (start,end,label)), sort."""
    seen, out = set(), []
    for s, e, lab in raw_spans:
        if trail_rule is not None:
            s, e = repair_span(text, s, e, lab, trail_rule, lead_rule or {})
        s, e = int(s), int(e)
        if not (0 <= s < e <= len(text)):
            continue
        key = (s, e, lab)
        if key in seen:
            continue
        seen.add(key)
        out.append([s, e, lab])
    out.sort()
    return out


# ---------------- strict scorer (same arithmetic as evaluate_phase2.py) + error breakdown ----------------
def strict_scores(golds, preds, labels):
    stats = {l: [0, 0, 0] for l in labels}
    for did, g in golds.items():
        gs = {(x[0], x[1], x[2]) for x in g}
        ps = {(x[0], x[1], x[2]) for x in preds.get(did, [])}
        for l in labels:
            a = {x for x in gs if x[2] == l}; b = {x for x in ps if x[2] == l}
            stats[l][0] += len(a & b); stats[l][1] += len(b - a); stats[l][2] += len(a - b)
    per = {}
    for l, (tp, fp, fn) in stats.items():
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        per[l] = dict(P=p, R=r, F1=f, tp=tp, fp=fp, fn=fn)
    n = len(labels)
    return dict(macro_F1=sum(v["F1"] for v in per.values()) / n,
                macro_P=sum(v["P"] for v in per.values()) / n,
                macro_R=sum(v["R"] for v in per.values()) / n, per_label=per)


def error_breakdown(golds, preds):
    """Classify each wrong prediction: boundary error (overlaps a gold entity of the same label), label error (same
    exact span, other label), boundary+label, or spurious (overlaps no gold entity). Also gold entities missed."""
    from collections import Counter
    c = Counter()
    for did, g in golds.items():
        gs = {(x[0], x[1], x[2]) for x in g}
        for (s, e, l) in {(x[0], x[1], x[2]) for x in preds.get(did, [])}:
            if (s, e, l) in gs:
                c["correct"] += 1; continue
            same_span = [x for x in gs if x[0] == s and x[1] == e]
            over = [x for x in gs if x[0] < e and s < x[1]]
            if same_span:
                c["label_error"] += 1
            elif any(x[2] == l for x in over):
                c["boundary_error"] += 1
            elif over:
                c["boundary_and_label_error"] += 1
            else:
                c["spurious"] += 1
        ps = {(x[0], x[1]) for x in preds.get(did, [])}
        for x in gs:
            if not any(p[0] < x[1] and x[0] < p[1] for p in ps):
                c["missed_entirely"] += 1
    return dict(c)


# ---------------- sliding windows ----------------
def make_windows(n_tokens, window, stride):
    """Start indices of windows of length `window` every `stride` tokens, the last one flush with the end."""
    if n_tokens <= window:
        return [0]
    starts = list(range(0, n_tokens - window + 1, stride))
    if starts[-1] + window < n_tokens:
        starts.append(n_tokens - window)
    return starts


## 6. Phase-1 model code (copied unchanged from mor-gqa-rope (Phase-1 notebook, cells 'Building blocks' + 'The model'))

In [6]:
NORM_TYPE = "layernorm"; NORM_POSITION = "pre"   # as in the Phase-1 run
CONTEXT_LEN = 2048

class RMSNorm(nn.Module):
    """Like LayerNorm, but simpler: it only rescales by the root-mean-square
    of the values, it does not re-center the mean. Cheaper, and used in
    models like LLaMA."""

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight


def make_norm(dim, norm_type):
    """norm_type is "layernorm", "rmsnorm" or "deepnorm".
    deepnorm reuses a plain LayerNorm -- what makes deepnorm special isn't
    the norm formula, it's the residual scaling (alpha) and weight init
    (beta) we apply around it below."""
    if norm_type == "rmsnorm":
        return RMSNorm(dim)
    else:  # "layernorm" or "deepnorm"
        return nn.LayerNorm(dim)


def deepnorm_alpha(n_layers):
    """Residual scaling factor from the DeepNet paper: alpha = (2N)^0.25."""
    return (2.0 * n_layers) ** 0.25


def deepnorm_init(output_projections, n_layers):
    """Scales the init of a sub-layer's OUTPUT projection(s) (e.g. attention
    out_proj, the feed-forward's second linear) by beta = (8N)^-0.25, so a
    deep post-norm stack stays stable to train. Only used when
    NORM_TYPE == "deepnorm"."""
    beta = (8.0 * n_layers) ** -0.25
    for module in output_projections:
        nn.init.xavier_normal_(module.weight, gain=beta)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

import math

# ---- Settings specific to this notebook ------------------------------
GQA_GROUP_SIZE = 4      # query heads that share one key/value head. 4 is exactly
                        # what ATTENTION_TYPE = "gqa" meant in the original notebooks.
CACHE_HEADROOM = 512    # slots reserved inside CONTEXT_LEN for GENERATED tokens.
                        # The prompt is cut to its last CONTEXT_LEN - CACHE_HEADROOM
                        # tokens; if generation fills the cache, the window is
                        # re-prefilled (see the markdown above for why). With
                        # CONTEXT_LEN = 2048 that leaves room for a 1536-token prompt
                        # (a 512-word legal prompt is roughly 700-1000 tokens) AND
                        # ~512 generated tokens, so in practice nothing is cut.
USE_NATIVE_GQA = False  # decided automatically by a micro-benchmark in the next
                        # section (True = pass enable_gqa=True to SDPA instead of
                        # expanding K/V with repeat_interleave)
POSITIONAL_ENCODING = "rope"   # "rope" | "learned"
                        # rope    = rotary embeddings: Q and K are rotated by a
                        #           position-dependent angle inside attention, so
                        #           attention scores depend on RELATIVE distance
                        #           (RoFormer, Su et al. 2021). No position table.
                        # learned = a trainable CONTEXT_LEN-row position table added to
                        #           the token embeddings (GPT-2 style, what the original
                        #           decoder-only notebook used).
ROPE_BASE = 10000.0     # RoPE frequency base: theta_i = ROPE_BASE ** (-2i / head_dim)
assert POSITIONAL_ENCODING in ("rope", "learned"), "POSITIONAL_ENCODING must be 'rope' or 'learned'"


def build_rope_cache(max_seq_len, head_dim, base=10000.0):
    """cos/sin tables for RoPE, each (max_seq_len, head_dim). Split-half
    convention: channel i is paired with channel i + head_dim/2, so the angle
    table is cat([freqs, freqs]) (NOT repeat_interleave, which pairs neighbours
    instead). Same construction as the RoPE written for Assignment 3."""
    assert head_dim % 2 == 0, "RoPE requires an even head dimension"
    inv_freq = 1.0 / (base ** (2 * torch.arange(0, head_dim // 2).float() / head_dim))
    positions = torch.arange(0, max_seq_len).float()
    freqs = torch.outer(positions, inv_freq)              # (max_seq_len, head_dim // 2)
    angles = torch.cat([freqs, freqs], dim=-1)            # (max_seq_len, head_dim)
    return angles.cos(), angles.sin()


def rotate_half(x):
    """Split the last dim in half and return (-x2, x1): a 90-degree rotation step."""
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    """x_rotated = x * cos + rotate_half(x) * sin. x: (..., head_dim); cos/sin
    already broadcastable against x (the model gathers them per position). The
    arithmetic is done in fp32 (even under fp16 autocast) and cast back, so the
    rotation stays accurate and q/k keep the same dtype as v for SDPA."""
    xf = x.float()
    return (xf * cos + rotate_half(xf) * sin).to(x.dtype)


# =====================================================================================================================
#  LegalMoR : Mixture-of-Recursions with full-context deep passes, causal threshold routing and pass/depth embeddings.
#
#  BASE PAPER   Mixture-of-Recursions (Bae et al., NeurIPS 2025, arXiv 2507.10524). One shared block of layers is applied several times
#               (Middle-Cycle: unique first layer, shared middle block, unique last layer). A tiny router decides PER TOKEN whether it gets
#               another pass, so easy tokens stop early and hard tokens get more depth; skipped tokens keep their state.
#  OUR CHANGES  (each has a switch below so it can be ablated)
#    1. FULL-CONTEXT DEEP PASSES (MOR_FULL_CONTEXT): in the paper a deep pass attends only among the tokens still selected, so a hard token at
#       pass 3 sees about a third of the sequence -- and the tokens it cannot see are exactly the easy ones ("Section", "the appellant") it often
#       needs. Here only the selected tokens send queries and go through the FFN, but EVERY token supplies keys/values from its latest state
#       (a skipped token's frozen state). Deep passes keep the whole context; the FFN saving is unchanged.
#    2. CAUSAL THRESHOLD ROUTING (MOR_ROUTING): the paper's expert-choice router keeps the top-k scores of the whole sequence, so a token's
#       fate depends on later tokens (a leak, and impossible to reproduce while generating one token at a time). Here a token is kept iff its
#       OWN score beats a threshold tau_r; tau_r is a running quantile estimate (updated after each training step, never learned) that keeps
#       the intended share of tokens on average. Training and generation use the same rule; nothing looks at other tokens.
#    3. PASS + DEPTH EMBEDDINGS (MOR_PASS_EMBED / MOR_DEPTH_EMBED): the shared block cannot tell which pass it is on, and the unique last layer
#       receives rows with 1, 2 or 3 passes without knowing which. A learned vector per pass is added to the block input, and a learned
#       vector per reached depth is added before the last layer. Both start at zero.
#    4. RELAXED SHARING (MOR_SHARE_FFN = False): attention is shared across passes, but every pass has its own FFN. Compute is identical (the
#       same rows are processed); only the stored weights grow (~92M -> ~121M). FFNs act as key-value memories, so this adds storage for
#       facts and phrases where small models are short of capacity -- the paper itself found MoR behind a plain Transformer at ~135M.
#  Sizes: d_model 768, 12 heads (GQA, 3 KV heads), FFN 3072; layout 1 + 3 shared layers x 3 passes + 1 = up to 11 layers deep.
# =====================================================================================================================

# ---- Settings of the recursion (each is an ablation switch) ------------------------------------------------------
MOR_SHARED_LAYERS = 3        # layers in the shared middle block
MOR_RECURSIONS = 3           # passes through the shared block
MOR_KEEP = (1.0, 7 / 8, 6 / 8)   # target share of ALL tokens processed at pass 1, 2, 3. Gentle skipping on purpose (the paper uses 1, 2/3, 1/3): the per-pass FFNs see most tokens and little is dropped; compute saving is small
MOR_ROUTING = "threshold"    # "threshold" = our causal router | "none" = every token gets every pass (fixed-depth recursive baseline)
MOR_FULL_CONTEXT = True      # change 1. False = the paper's recursion-wise attention (deep passes see only the tokens still selected)
MOR_PASS_EMBED = True        # change 3a
MOR_DEPTH_EMBED = True       # change 3b
MOR_TAU_MOMENTUM = 0.1       # how fast each threshold follows the batch quantile. The router keeps pushing scores up, so a slow threshold lags: in 600-step CPU tests 0.01 let up to 53% of tokens reach full depth (target 33%); 0.1 kept 28-38%
MOR_SHARE_FFN = False        # False = each pass has its OWN FFN (attention stays shared): ~28M more parameters, same compute ("relaxed sharing")
                             # True  = the paper's full sharing (attention AND FFN reused at every pass)
MOR_QUERY_GATHER = "auto"    # speed: True = deep passes compute attention only for the selected queries (fewer FLOPs, needs an explicit mask);
                             # False = attention for every query, unselected rows discarded (fused causal kernel). "auto" = time both on the GPU, keep the faster
assert MOR_ROUTING in ("threshold", "none") and MOR_QUERY_GATHER in ("auto", True, False)
assert len(MOR_KEEP) == MOR_RECURSIONS and MOR_KEEP[0] == 1.0 and all(MOR_KEEP[i] >= MOR_KEEP[i + 1] > 0 for i in range(MOR_RECURSIONS - 1))
assert NORM_POSITION == "pre" and NORM_TYPE != "deepnorm", "the recursion block is pre-norm only"


class KVCache:
    """Pre-allocated key/value storage for EVERY layer application (first layer, each shared layer at each pass, last layer -- a shared layer has
    separate K/V per pass because its input differs), written in place. Also the per-pass selection history, which the paper-style attention
    (MOR_FULL_CONTEXT = False) needs to know which earlier tokens a deep pass may look at. `length` = filled positions (same for every row)."""

    def __init__(self, n_layers, batch_size, n_kv_heads, max_len, head_dim, device, dtype):
        shape = (batch_size, n_kv_heads, max_len, head_dim)
        self.k = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(n_layers)]
        self.v = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(n_layers)]
        self.sel = torch.ones(MOR_RECURSIONS, batch_size, max_len, dtype=torch.bool, device=device)
        self.length = 0
        self.max_len = max_len

    def reset(self):
        self.length = 0

    def num_bytes(self):
        return sum(t.numel() * t.element_size() for t in self.k + self.v + [self.sel])



class CachedGQAttention(nn.Module):
    """Grouped-query self-attention (causal) with two entry points:
      forward()      -- the parallel form used for training and for PREFILL.
                        Optionally also stores this call's keys/values in a cache.
      decode_step()  -- ONE new token, reading/writing the cache.
    Parameter names (q_proj/k_proj/v_proj/out_proj) match the original
    MultiHeadAttention with ATTENTION_TYPE="gqa", so checkpoints are interchangeable."""

    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.group_size = n_heads // n_kv_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.out_proj = nn.Linear(n_heads * self.head_dim, d_model)

    def forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
        """x: (batch, seq, d_model). key_padding_mask (optional): (batch, seq)
        bool, True = real (non-pad) position; leave None when nothing is
        padded so the fused kernels can run mask-free. If k_cache/v_cache are
        given, this call's (un-expanded) keys/values are written to
        positions [0:seq] of them -- that is the PREFILL step. rope (optional):
        (cos, sin), each (batch or 1, 1, seq, head_dim), gathered from the
        model's RoPE tables -- None when POSITIONAL_ENCODING == "learned"."""
        batch_size, seq_len, _ = x.shape
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)

        if rope is not None:
            # rotate BEFORE caching: the cache stores already-rotated keys (each
            # key is rotated by its own position exactly once, when it is written)
            q = apply_rope(q, *rope)
            k = apply_rope(k, *rope)

        if k_cache is not None:
            k_cache[:, :, :seq_len] = k.to(k_cache.dtype)
            v_cache[:, :, :seq_len] = v.to(v_cache.dtype)

        attn_mask = None
        if key_padding_mask is not None:
            positions = torch.arange(seq_len, device=x.device)
            causal = positions.unsqueeze(1) >= positions.unsqueeze(0)               # (seq, seq)
            allowed = causal.unsqueeze(0) & key_padding_mask.unsqueeze(1)           # (batch, seq, seq)
            # a query may always see itself -- guarantees no row is fully
            # masked (which would produce NaNs for pad-position queries)
            allowed = allowed | torch.eye(seq_len, dtype=torch.bool, device=x.device).unsqueeze(0)
            attn_mask = allowed.unsqueeze(1)                                        # (batch, 1, seq, seq)

        use_causal_flag = attn_mask is None
        if USE_NATIVE_GQA and self.group_size > 1:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                                 is_causal=use_causal_flag, enable_gqa=True)
        else:
            if self.group_size > 1:
                k = k.repeat_interleave(self.group_size, dim=1)
                v = v.repeat_interleave(self.group_size, dim=1)
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                                 is_causal=use_causal_flag)

        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.n_heads * self.head_dim)
        return self.out_proj(out)

    def decode_step(self, x_t, k_cache, v_cache, cache_len, key_mask=None, rope=None):
        """x_t: (batch, d_model) -- ONE new position per row. Writes its K/V at
        index cache_len and attends over cache positions 0..cache_len
        (inclusive). key_mask (optional): (batch, cache_len + 1) bool, True =
        real key; None when the batch has no padding. rope (optional): (cos,
        sin), each (batch, head_dim) for this step's position."""
        batch_size = x_t.shape[0]

        # Fold the `group_size` query heads that share one K/V head into the
        # query-LENGTH axis: (batch, n_kv_heads, group_size, head_dim). The
        # cache is then used as-is (never expanded), and because the new token
        # is the last position no causal mask is needed.
        q = self.q_proj(x_t).view(batch_size, self.n_kv_heads, self.group_size, self.head_dim)
        k_new = self.k_proj(x_t).view(batch_size, self.n_kv_heads, self.head_dim)
        v_new = self.v_proj(x_t).view(batch_size, self.n_kv_heads, self.head_dim)

        if rope is not None:
            cos, sin = rope
            q = apply_rope(q, cos.view(batch_size, 1, 1, -1), sin.view(batch_size, 1, 1, -1))
            k_new = apply_rope(k_new, cos.view(batch_size, 1, -1), sin.view(batch_size, 1, -1))

        k_cache[:, :, cache_len] = k_new.to(k_cache.dtype)
        v_cache[:, :, cache_len] = v_new.to(v_cache.dtype)
        k = k_cache[:, :, :cache_len + 1]
        v = v_cache[:, :, :cache_len + 1]
        if q.dtype != k.dtype:
            q = q.to(k.dtype)

        attn_mask = None if key_mask is None else key_mask.view(batch_size, 1, 1, -1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)   # (batch, n_kv, group, head_dim)
        out = out.reshape(batch_size, self.n_heads * self.head_dim).to(x_t.dtype)
        return self.out_proj(out)


class CachedDecoderBlock(nn.Module):
    """Same block as DecoderOnlyBlock (same norm / deepnorm wiring), with a
    decode_step() that mirrors forward() for one position."""

    def __init__(self, d_model, n_heads, d_ff, n_layers):
        super().__init__()
        self.norm_position = "post" if NORM_TYPE == "deepnorm" else NORM_POSITION

        n_kv_heads = max(1, n_heads // GQA_GROUP_SIZE)
        self.attn = CachedGQAttention(d_model, n_heads, n_kv_heads)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.norm1 = make_norm(d_model, NORM_TYPE)
        self.norm2 = make_norm(d_model, NORM_TYPE)

        if NORM_TYPE == "deepnorm":
            self.alpha = deepnorm_alpha(n_layers)
            deepnorm_init([self.attn.out_proj, self.ff[2]], n_layers)
        else:
            self.alpha = 1.0

    def forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
        if self.norm_position == "pre":
            x = x + self.attn(self.norm1(x), key_padding_mask=key_padding_mask,
                              k_cache=k_cache, v_cache=v_cache, rope=rope)
            x = x + self.ff(self.norm2(x))
        else:  # "post" (also deepnorm, with the alpha residual scale)
            x = self.norm1(x * self.alpha + self.attn(x, key_padding_mask=key_padding_mask,
                                                       k_cache=k_cache, v_cache=v_cache, rope=rope))
            x = self.norm2(x * self.alpha + self.ff(x))
        return x

    def decode_step(self, x_t, k_cache, v_cache, cache_len, key_mask=None, rope=None):
        """x_t: (batch, d_model). Norms and the FFN act per position, so only
        the attention part needs the cache."""
        if self.norm_position == "pre":
            x_t = x_t + self.attn.decode_step(self.norm1(x_t), k_cache, v_cache, cache_len, key_mask, rope)
            x_t = x_t + self.ff(self.norm2(x_t))
        else:
            x_t = self.norm1(x_t * self.alpha + self.attn.decode_step(x_t, k_cache, v_cache, cache_len, key_mask, rope))
            x_t = self.norm2(x_t * self.alpha + self.ff(x_t))
        return x_t


class CachedDecoderModel(nn.Module):
    """Mixture-of-Recursions decoder (GQA + RoPE), KV-cache aware, same entry points as the other notebooks: forward() / prefill() / decode_step().
    self.blocks is the UNROLLED execution order  [first, shared_0..shared_{S-1} (pass 1), ... (pass R), last]  -- the shared layers appear once per
    pass but are the SAME modules (parameters are counted and optimised once), and position i in this list is also KV-cache slot i.
    The n_layers argument is ignored: the layout comes from MOR_SHARED_LAYERS and MOR_RECURSIONS."""

    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, context_len):
        super().__init__()
        assert POSITIONAL_ENCODING == "rope", "this notebook is RoPE-only"
        self.token_emb = nn.Embedding(vocab_size, d_model)
        cos, sin = build_rope_cache(context_len, d_model // n_heads, ROPE_BASE)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        S, R = MOR_SHARED_LAYERS, MOR_RECURSIONS
        first = CachedDecoderBlock(d_model, n_heads, d_ff, 2 + S)
        shared = [CachedDecoderBlock(d_model, n_heads, d_ff, 2 + S) for _ in range(S)]
        passes = [shared]
        for _ in range(1, R):
            if MOR_SHARE_FFN:
                passes.append(shared)
            else:   # relaxed sharing: a fresh block per pass whose attention (and its norm) IS the pass-1 module; its FFN (and norm) is its own
                row = []
                for j in range(S):
                    b = CachedDecoderBlock(d_model, n_heads, d_ff, 2 + S)
                    b.attn, b.norm1 = shared[j].attn, shared[j].norm1
                    row.append(b)
                passes.append(row)
        last = CachedDecoderBlock(d_model, n_heads, d_ff, 2 + S)
        self.blocks = nn.ModuleList([first] + [passes[r][j] for r in range(R) for j in range(S)] + [last])
        self.query_gather = (MOR_QUERY_GATHER is True)     # "auto" is resolved by the speed-probe cell on the GPU
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.S, self.R = S, R
        self.routed = (MOR_ROUTING == "threshold")
        if self.routed:   # one router per pass that is routed (pass 1 takes every token)
            self.router_norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(R - 1)])
            self.router = nn.ModuleList([nn.Linear(d_model, 1) for _ in range(R - 1)])
            for lin in self.router:
                nn.init.normal_(lin.weight, std=0.02)
                nn.init.zeros_(lin.bias)
            self.register_buffer("tau", torch.zeros(R - 1))
            self.register_buffer("tau_ready", torch.zeros(R - 1, dtype=torch.bool))
            self._ready_py = [None] * (R - 1)   # python copy of tau_ready, read from the buffer once (avoids a GPU->CPU sync every step)
        self.pass_emb = nn.Parameter(torch.zeros(R, d_model)) if MOR_PASS_EMBED else None
        self.depth_emb = nn.Parameter(torch.zeros(R, d_model)) if MOR_DEPTH_EMBED else None

    def slot(self, r, j):
        """KV-cache slot / index in self.blocks of shared layer j at pass r (both 0-based)."""
        return 1 + r * self.S + j

    def _rope(self, position_ids):
        cos, sin = self.rope_cos[position_ids], self.rope_sin[position_ids]
        return (cos.unsqueeze(1), sin.unsqueeze(1))

    # ---- routing -------------------------------------------------------------------------------------------------------
    def _route(self, r, h, eligible):
        """Pass r >= 1 (0-based). h (...,d); eligible (...) bool = selected at the previous pass. Returns (selected, gate). The decision for a token
        uses ONLY its own score and the stored threshold, so it is causal. In training the threshold then moves towards the batch quantile that
        would keep the target share (used from the NEXT step on)."""
        z = self.router[r - 1](self.router_norm[r - 1](h)).squeeze(-1).float()
        tau = self.tau[r - 1].clone()          # decisions use the threshold from BEFORE this batch (no information from other tokens)
        if self.training:
            with torch.no_grad():   # no boolean indexing and no .item(): nothing here waits for the GPU
                q = torch.nanquantile(z.detach().masked_fill(~eligible, float("nan")).flatten(), 1.0 - MOR_KEEP[r] / MOR_KEEP[r - 1])
                old = self.tau[r - 1]
                if self._ready_py[r - 1] is None:
                    self._ready_py[r - 1] = bool(self.tau_ready[r - 1])
                new = old + MOR_TAU_MOMENTUM * (q - old) if self._ready_py[r - 1] else q
                self.tau[r - 1].copy_(torch.where(torch.isnan(q), old, new))
                self.tau_ready[r - 1] = True
                self._ready_py[r - 1] = True
        selected = eligible & (z > tau)
        gate = torch.sigmoid(z - tau)
        return selected, gate

    # ---- one pass through the shared block (parallel form) ------------------------------------------------------------
    def _pass(self, r, h, sel, key_padding_mask, rope, cache):
        """h (B,T,d). sel (B,T) bool or None (= every token). Only selected rows change; every row supplies keys/values (full context) unless
        MOR_FULL_CONTEXT is False, in which case keys are restricted to the selected rows (the paper's recursion-wise attention)."""
        x = h + self.pass_emb[r] if self.pass_emb is not None else h
        x_in = x
        kpm = key_padding_mask
        if sel is not None and not MOR_FULL_CONTEXT:
            kpm = sel if kpm is None else (kpm & sel)
        selm = None if sel is None else sel.unsqueeze(-1)
        for j in range(self.S):
            blk = self.blocks[self.slot(r, j)]
            kc, vc = (cache.k[self.slot(r, j)], cache.v[self.slot(r, j)]) if cache is not None else (None, None)
            if sel is not None and self.query_gather:
                a = self._attn_selected(blk.attn, blk.norm1(x), sel, kpm, rope, kc, vc)
            else:
                a = blk.attn(blk.norm1(x), key_padding_mask=kpm, k_cache=kc, v_cache=vc, rope=rope)
            if selm is None:
                x = x + a
                x = x + blk.ff(blk.norm2(x))
            else:
                x = x + a * selm.to(a.dtype)
                rows = x[sel]                                                    # (N, d): only the selected tokens go through the FFN
                x = x.masked_scatter(selm.expand_as(x), rows + blk.ff(blk.norm2(rows)))
        return x - x_in                                                          # the block's change (zero on unselected rows)

    def _attn_selected(self, att, xn, sel, key_mask, rope, kc, vc):
        """Same result as att(xn, ...) on the selected rows (zeros elsewhere), but queries are computed only for the selected tokens: each row's
        selected positions are gathered (in order, padded to the batch maximum), keys/values come from EVERY position (and are written to the
        cache exactly as the plain path does), and the causal mask uses the original positions."""
        B, T, D = xn.shape
        k = att.k_proj(xn).view(B, T, att.n_kv_heads, att.head_dim).transpose(1, 2)
        v = att.v_proj(xn).view(B, T, att.n_kv_heads, att.head_dim).transpose(1, 2)
        k = apply_rope(k, rope[0], rope[1])
        if kc is not None:
            kc[:, :, :T] = k.to(kc.dtype)
            vc[:, :, :T] = v.to(vc.dtype)
        counts = sel.sum(1)
        kmax = int(counts.max())
        if kmax == 0:
            return att.out_proj(xn.new_zeros(B, T, att.n_heads * att.head_dim)) * 0
        order = torch.sort(sel.to(torch.int8), dim=1, descending=True, stable=True).indices[:, :kmax]     # selected positions first, in order
        valid = torch.arange(kmax, device=xn.device).unsqueeze(0) < counts.unsqueeze(1)
        q = att.q_proj(xn.gather(1, order.unsqueeze(-1).expand(-1, -1, D))).view(B, kmax, att.n_heads, att.head_dim).transpose(1, 2)
        cos, sin = rope[0].expand(B, -1, -1, -1), rope[1].expand(B, -1, -1, -1)
        idx = order.unsqueeze(1).unsqueeze(-1).expand(-1, 1, -1, cos.shape[-1])
        q = apply_rope(q, cos.gather(2, idx), sin.gather(2, idx))
        jpos = torch.arange(T, device=xn.device).view(1, 1, T)
        allowed = jpos <= order.unsqueeze(-1)                                    # (B, kmax, T): causal on ORIGINAL positions
        if key_mask is not None:
            allowed = allowed & key_mask.unsqueeze(1)
        allowed = allowed | (jpos == order.unsqueeze(-1)) | ~valid.unsqueeze(-1)  # a query always sees itself; padded query rows are discarded
        if att.group_size > 1:
            k = k.repeat_interleave(att.group_size, dim=1)
            v = v.repeat_interleave(att.group_size, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed.unsqueeze(1))
        out = att.out_proj(out.transpose(1, 2).reshape(B, kmax, att.n_heads * att.head_dim))
        out = out * valid.unsqueeze(-1).to(out.dtype)
        return out.new_zeros(B, T, out.shape[-1]).scatter_add(1, order.unsqueeze(-1).expand_as(out), out)

    def _trunk(self, x_ids, key_padding_mask, position_ids, cache=None):
        h = self.token_emb(x_ids)
        rope = self._rope(position_ids)
        kc = (cache.k[0], cache.v[0]) if cache is not None else (None, None)
        h = self.blocks[0](h, key_padding_mask=key_padding_mask, k_cache=kc[0], v_cache=kc[1], rope=rope)
        B, T = x_ids.shape
        sel = None
        depth = torch.ones(B, T, dtype=torch.long, device=x_ids.device)
        for r in range(self.R):
            if r == 0 or not self.routed:
                gate = None
                if self.routed:
                    sel = torch.ones(B, T, dtype=torch.bool, device=x_ids.device)
            else:
                sel, gate = self._route(r, h, sel)
                depth = depth + sel.long()
            if cache is not None and sel is not None:
                cache.sel[r, :, :T] = sel
            pass_sel = None if (r == 0 or not self.routed) else sel     # pass 1 takes every token: no gather needed
            delta = self._pass(r, h, pass_sel, key_padding_mask, rope, cache)
            h = h + (delta if gate is None else gate.unsqueeze(-1).to(delta.dtype) * delta)
        if not self.routed:
            depth = depth + (self.R - 1)
        if self.depth_emb is not None:
            h = h + self.depth_emb[depth - 1]
        kc = (cache.k[-1], cache.v[-1]) if cache is not None else (None, None)
        h = self.blocks[-1](h, key_padding_mask=key_padding_mask, k_cache=kc[0], v_cache=kc[1], rope=rope)
        self._last_depth = depth
        return h

    # ---- training / full recompute ---------------------------------------------------------------------------------
    def forward(self, x, key_padding_mask=None, position_ids=None, last_position_only=False):
        B, T = x.shape
        if position_ids is None:
            position_ids = torch.arange(T, device=x.device).unsqueeze(0)
        h = self._trunk(x, key_padding_mask, position_ids)
        if last_position_only:
            h = h[:, -1, :]
        return self.fc_out(h)

    # ---- generation --------------------------------------------------------------------------------------------------
    def prefill(self, x, pad_lens, cache):
        B, P = x.shape
        assert P <= cache.max_len, "prompt window longer than the cache"
        pad_t = torch.tensor(pad_lens, dtype=torch.long, device=x.device)
        arange = torch.arange(P, device=x.device).unsqueeze(0)
        position_ids = (arange - pad_t.unsqueeze(1)).clamp(min=0)
        key_padding_mask = (arange >= pad_t.unsqueeze(1)) if any(p > 0 for p in pad_lens) else None
        h = self._trunk(x, key_padding_mask, position_ids, cache=cache)
        cache.length = P
        return self.fc_out(h[:, -1, :])

    def decode_step(self, token_ids, pad_t, has_padding, cache):
        """ONE new token per row. A token that is NOT selected at a pass still writes its keys/values for that pass (from its frozen state), so
        later tokens can attend to it -- exactly what the parallel form does."""
        L = cache.length
        assert L < cache.max_len, "cache is full -- re-prefill before decoding further"
        B = token_ids.shape[0]
        dev = token_ids.device
        position_ids = (L - pad_t).clamp(min=0)
        real = torch.arange(L + 1, device=dev).unsqueeze(0) >= pad_t.unsqueeze(1) if has_padding else None
        rope = (self.rope_cos[position_ids], self.rope_sin[position_ids])
        h = self.token_emb(token_ids)
        h = self.blocks[0].decode_step(h, cache.k[0], cache.v[0], L, real, rope)
        sel = torch.ones(B, dtype=torch.bool, device=dev)
        depth = torch.ones(B, dtype=torch.long, device=dev)
        for r in range(self.R):
            gate = None
            if r > 0 and self.routed:
                sel, gate = self._route(r, h, sel)
                depth = depth + sel.long()
            cache.sel[r, :, L] = sel
            key_mask = real
            if self.routed and not MOR_FULL_CONTEXT:
                key_mask = cache.sel[r, :, :L + 1].clone() if real is None else (cache.sel[r, :, :L + 1] & real)
                key_mask[:, L] = True                             # a query always sees itself (as in the parallel form)
            x = h + self.pass_emb[r] if self.pass_emb is not None else h
            x_in = x
            for j in range(self.S):
                blk = self.blocks[self.slot(r, j)]
                x_new = blk.decode_step(x, cache.k[self.slot(r, j)], cache.v[self.slot(r, j)], L, key_mask, rope)
                x = torch.where(sel.unsqueeze(-1), x_new, x) if self.routed else x_new
            delta = x - x_in
            h = h + (delta if gate is None else gate.unsqueeze(-1).to(delta.dtype) * delta)
        if not self.routed:
            depth = depth + (self.R - 1)
        if self.depth_emb is not None:
            h = h + self.depth_emb[depth - 1]
        h = self.blocks[-1].decode_step(h, cache.k[-1], cache.v[-1], L, real, rope)
        cache.length = L + 1
        return self.fc_out(h)

    # ---- diagnostics for the report ----------------------------------------------------------------------------------
    @torch.no_grad()
    def diagnostics(self, x):
        """x (B,T) token ids. Returns per-token depth (B,T), the share of tokens at each depth, the thresholds and the mean gate of the selected
        tokens at each routed pass."""
        T = x.shape[1]
        gates = []
        if self.routed:
            orig = self._route
            def spy(r, h, eligible, _orig=orig):
                s, g = _orig(r, h, eligible)
                gates.append(g[s].float().mean().item() if s.any() else float("nan"))
                return s, g
            self._route = spy
        try:
            self._trunk(x, None, torch.arange(T, device=x.device).unsqueeze(0))
        finally:
            if self.routed:
                del self._route
        depth = self._last_depth
        out = {"depth": depth, "depth_share": [(depth == d + 1).float().mean().item() for d in range(self.R)],
               "mean_gate_selected": gates}
        if self.routed:
            out["tau"] = self.tau.tolist()
        return out


## 7. NER backbone wrapper + loading the Phase-1 model

In [7]:
# ===================== NER backbone wrappers for the other Phase-1 architectures =====================
# The Phase-1 model code above is UNCHANGED. Each wrapper only exposes the hidden states that feed the NER head.

def _unique(mods):
    seen, out = set(), []
    for m in mods:
        if id(m) not in seen:
            seen.add(id(m)); out.append(m)
    return out


class ForwardBackbone(nn.Module):
    """For decoder models with their own forward logic (MoR recursion + routing, Boundary Canon): the model's own forward()
    is called with the vocabulary projection replaced by the identity, so it returns its last hidden states."""

    def __init__(self, lm):
        super().__init__()
        self.original_param_count = sum(p.numel() for p in lm.parameters())
        self.lm = lm
        self.causal = True
        if isinstance(lm.fc_out, nn.Sequential):
            self.final_norm = lm.fc_out[0]
        else:
            self.final_norm = nn.Identity()
        lm.fc_out = nn.Identity()
        self.d_model = lm.token_emb.embedding_dim

    def layer_list(self):
        return _unique(list(self.lm.blocks))

    def final_modules_params(self):
        return list(self.final_norm.parameters())

    def embedding_weight(self):
        return self.lm.token_emb.weight

    def adapter_sites(self, kind):
        blocks = _unique(list(self.lm.blocks))
        if kind == "houlsby":
            return [(m, self.d_model) for m in _unique([x for b in blocks for x in (b.attn, b.ff)])]
        if kind == "pfeiffer":
            return [(m, self.d_model) for m in _unique([b.ff for b in blocks])]
        raise ValueError(kind)

    def enable_prefix(self, P, hidden=0):
        raise NotImplementedError("prefix tuning is only wired for the plain GQA/MoE decoders")

    def features(self, ids, mask, soft_prompt=None):
        assert soft_prompt is None, "prompt tuning is only wired for the plain GQA/MoE decoders"
        # right padding + causal model: real tokens never see the pads, so no mask is passed (as in Phase-1 training)
        return self.final_norm(self.lm(ids))


class LSTMBackbone(nn.Module):
    """The Phase-1 3-layer LSTM, split into one nn.LSTM per layer (weights copied exactly) so that adapters can sit between
    layers and partial fine-tuning can pick the top layers. Hidden states of the last layer (LSTM_HIDDEN wide) feed the head."""

    def __init__(self, lm):
        super().__init__()
        self.original_param_count = sum(p.numel() for p in lm.parameters())
        self.emb = lm.emb
        L, H = lm.lstm.num_layers, lm.lstm.hidden_size
        self.layers = nn.ModuleList()
        for k in range(L):
            layer = nn.LSTM(lm.lstm.input_size if k == 0 else H, H, num_layers=1, batch_first=True)
            with torch.no_grad():
                for n in ("weight_ih", "weight_hh", "bias_ih", "bias_hh"):
                    getattr(layer, f"{n}_l0").copy_(getattr(lm.lstm, f"{n}_l{k}"))
            self.layers.append(layer)
        self.taps = nn.ModuleList([nn.Identity() for _ in range(L)])    # adapter attachment points (after each layer)
        self.d_model = H
        self.causal = True

    def layer_list(self):
        return list(self.layers)

    def final_modules_params(self):
        return []

    def embedding_weight(self):
        return self.emb.weight

    def adapter_sites(self, kind):
        if kind == "houlsby":
            return [(t, self.d_model) for t in self.taps]
        if kind == "pfeiffer":
            return [(self.taps[-1], self.d_model)]
        raise ValueError(kind)

    def enable_prefix(self, P, hidden=0):
        raise NotImplementedError("prefix tuning does not apply to an LSTM")

    def features(self, ids, mask, soft_prompt=None):
        assert soft_prompt is None, "prompt tuning is not wired for the LSTM"
        h = self.emb(ids)
        for layer, tap in zip(self.layers, self.taps):
            layer.flatten_parameters()
            h, _ = layer(h)
            h = tap(h)
        return h


class GDNEncoderBackbone(nn.Module):
    """The ENCODER of the Phase-1 Gated-DeltaNet encoder-decoder (6 bidirectional GDN blocks + final norm). Its decoder and
    output head are dropped. Bidirectional by construction (a forward and a backward GDN mixer in every block)."""

    def __init__(self, lm):
        super().__init__()
        self.original_param_count = sum(p.numel() for p in lm.parameters())
        self.lm = lm
        lm.dec_blocks = nn.ModuleList()
        lm.fc_out = nn.Identity()
        lm.enc_mix_logits = None            # only used by the decoder; also stops it storing every layer's states
        self.causal = False
        self.d_model = lm.token_emb.embedding_dim

    def layer_list(self):
        return list(self.lm.enc_blocks)

    def final_modules_params(self):
        return list(self.lm.enc_norm.parameters())

    def embedding_weight(self):
        return self.lm.token_emb.weight

    def adapter_sites(self, kind):
        blocks = list(self.lm.enc_blocks)
        if kind == "houlsby":
            return [(m, self.d_model) for b in blocks for m in (b.fwd, b.bwd, b.ff)]
        if kind == "pfeiffer":
            return [(b.ff, self.d_model) for b in blocks]
        raise ValueError(kind)

    def enable_prefix(self, P, hidden=0):
        raise NotImplementedError("prefix tuning does not apply to the GDN encoder")

    def features(self, ids, mask, soft_prompt=None):
        assert soft_prompt is None, "prompt tuning is not wired for the GDN encoder"
        B, T = ids.shape
        pos = torch.arange(T, device=ids.device).unsqueeze(0).expand(B, T)
        enc_mask = None if bool(mask.all()) else mask            # the backward mixer must not read right-padding
        h, _ = self.lm.encode(ids, enc_mask, pos)
        return h


@torch.no_grad()
def gdn_sanity_loss(lm, docs, device, n_windows=6, enc_len=768, dec_len=256):
    """Phase-1 objective of the encoder-decoder on NER text: encoder reads enc_len tokens, decoder predicts the next dec_len."""
    lm.eval().to(device)
    tot, cnt = 0.0, 0
    for d in list(docs.values())[:n_windows]:
        ids = torch.tensor(d["ids"][:enc_len + dec_len], dtype=torch.long, device=device).unsqueeze(0)
        if ids.shape[1] < enc_len + 8:
            continue
        enc, rest = ids[:, :enc_len], ids[:, enc_len - 1:]
        dec_in, tgt = rest[:, :-1], rest[:, 1:]
        enc_pos = torch.arange(enc.shape[1], device=device).unsqueeze(0)
        dec_pos = (enc_len - 1 + torch.arange(dec_in.shape[1], device=device)).unsqueeze(0)
        logits = lm(enc, dec_in, None, enc_pos, dec_pos).float()
        tot += F.cross_entropy(logits.view(-1, logits.shape[-1]), tgt.reshape(-1), reduction="sum").item()
        cnt += tgt.numel()
    return tot / max(cnt, 1)


@torch.no_grad()
def lm_sanity_loss(lm, docs, device, n_windows=6, length=1024):
    """Next-token loss of the loaded Phase-1 model on a few NER-document windows (BEFORE the LM head is dropped).
    A correctly loaded model gives roughly its Phase-1 held-out loss; random weights give ~ln(vocab) = 10.6."""
    lm.eval().to(device)
    tot, cnt = 0.0, 0
    for d in list(docs.values())[:n_windows]:
        ids = torch.tensor(d["ids"][:length + 1], dtype=torch.long, device=device).unsqueeze(0)
        if ids.shape[1] < 2:
            continue
        logits = lm(ids[:, :-1]).float()
        tot += F.cross_entropy(logits.view(-1, logits.shape[-1]), ids[:, 1:].reshape(-1), reduction="sum").item()
        cnt += ids.shape[1] - 1
    return tot / max(cnt, 1)


MOR_FREEZE_ROUTING = True    # keep the Phase-1 routing thresholds (tau) FIXED while fine-tuning. In Phase 1 they are updated inside the
                             # forward pass in training mode (a running quantile, not learned); on NER they drift with the new data, so
                             # tokens change depth between steps, and one fp16 overflow turns them into NaN for good (seen on Kaggle).
_MOR_ORIG_ROUTE = CachedDecoderModel._route


def _mor_route_frozen(self, r, h, eligible):
    """Same decision rule as Phase-1 _route (own score vs threshold -> selected, sigmoid gate), but never updates the threshold."""
    z = self.router[r - 1](self.router_norm[r - 1](h)).squeeze(-1).float()
    tau = self.tau[r - 1].clone()
    return eligible & (z > tau), torch.sigmoid(z - tau)


CachedDecoderModel._route = _mor_route_frozen if MOR_FREEZE_ROUTING else _MOR_ORIG_ROUTE


def build_phase1_lm():
    """The Phase-1 MoR model exactly as trained, with its trained weights (incl. the routing thresholds)."""
    torch.manual_seed(0)
    lm = CachedDecoderModel(VOCAB, d_model=768, n_heads=12, n_layers=8, d_ff=3072, context_len=CONTEXT_LEN)
    lm.load_state_dict(PHASE1_STATE, strict=True)
    return lm


def make_backbone(lm, C=None):
    return ForwardBackbone(lm)


def phase1_sanity(lm, docs):
    return lm_sanity_loss(lm, docs, device)


## 8. NER engine: head, adaptation methods, training, inference

In [8]:
# ===================== Phase 2 L-NER engine: backbone wrapper, NER head, adaptation methods, training, inference =====================
# Works with any Phase-1 model wrapped as a "backbone" exposing  features(ids, mask, soft_prompt) -> (B, T, d_model).
import math, time, random, json, os, copy
import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------------------------------------------------
# 1. Documents -> token ids, character spans, BIO tags; windows
# ----------------------------------------------------------------------------------------------------------------------
def prepare_docs(tok, texts, golds, labels, mode):
    docs = {}
    for did, text in texts.items():
        ids, spans = encode_with_offsets(tok, text, mode)
        tags = gold_to_token_tags(spans, golds[did], labels)[0] if golds is not None else None
        docs[did] = dict(text=text, ids=ids, spans=spans, tags=tags)
    return docs


def window_items(docs, window, stride):
    items = []
    for did, d in docs.items():
        n = len(d["ids"])
        if n == 0:
            continue
        for s in make_windows(n, window, stride):
            items.append((did, s))
    return items


def collate(docs, items, window, pad_id=0, with_labels=True):
    L = max(min(window, len(docs[did]["ids"]) - s) for did, s in items)
    B = len(items)
    ids = torch.full((B, L), pad_id, dtype=torch.long)
    mask = torch.zeros((B, L), dtype=torch.bool)
    labels = torch.full((B, L), IGNORE, dtype=torch.long) if with_labels else None
    for b, (did, s) in enumerate(items):
        seg = docs[did]["ids"][s:s + window]
        n = len(seg)
        ids[b, :n] = torch.tensor(seg, dtype=torch.long)
        mask[b, :n] = True
        if with_labels:
            labels[b, :n] = torch.tensor(docs[did]["tags"][s:s + window], dtype=torch.long)
    return ids, mask, labels


# ----------------------------------------------------------------------------------------------------------------------
# 2. NER model = backbone + (optional soft prompt) + token-classification head
# ----------------------------------------------------------------------------------------------------------------------
class BiLSTMLayer(nn.Module):
    """A bidirectional LSTM over the backbone's hidden states (a prediction-head option). It gives every token the
    context to its RIGHT inside the window without changing the pretrained (causal) layers. Padding is packed away
    so the backward direction never reads pad positions."""

    def __init__(self, d_in, hidden):
        super().__init__()
        self.lstm = nn.LSTM(d_in, hidden, batch_first=True, bidirectional=True)
        self.out_dim = 2 * hidden

    def forward(self, h, mask):
        lengths = mask.sum(1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(h.float(), lengths, batch_first=True, enforce_sorted=False)
        with torch.autocast(device_type=h.device.type, enabled=False):
            out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=h.shape[1])
        return out


class NERModel(nn.Module):
    def __init__(self, backbone, d_model, n_tags, head_type="linear", head_dropout=0.1, head_norm=False, lstm_hidden=256):
        super().__init__()
        self.backbone = backbone
        self.soft_prompt = None                          # set by the prompt-tuning technique
        self.head_type = head_type
        self.bilstm = None
        layers = [nn.Dropout(head_dropout)]
        if head_norm:
            layers.append(nn.LayerNorm(d_model))
        if head_type == "linear":
            layers.append(nn.Linear(d_model, n_tags))
        elif head_type == "mlp":
            layers += [nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(head_dropout), nn.Linear(d_model, n_tags)]
        elif head_type == "bilstm":
            self.bilstm = BiLSTMLayer(d_model, lstm_hidden)
            layers += [nn.Linear(self.bilstm.out_dim, n_tags)]
        else:
            raise ValueError(head_type)
        self.head = nn.Sequential(*layers)

    def head_parameters(self):
        ps = list(self.head.parameters())
        if self.bilstm is not None:
            ps += list(self.bilstm.parameters())
        return ps

    def forward(self, ids, mask):
        h = self.backbone.features(ids, mask, self.soft_prompt)
        if self.bilstm is not None:
            h = self.bilstm(self.head[0](h), mask)
            return self.head[1:](h)
        return self.head(h)


# ----------------------------------------------------------------------------------------------------------------------
# 3. Adaptation methods
# ----------------------------------------------------------------------------------------------------------------------
class LoRALinear(nn.Module):
    """LoRA (Hu et al. 2022): y = W0 x + b + (alpha/r) * B A x.  A: (r, in) random, B: (out, r) zeros -> starts == W0."""

    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base = base
        self.r, self.scale = r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)


_ROT_IDX = {}


def _row_rotation_index(d, device):
    key = (d, str(device))
    if key not in _ROT_IDX:
        i = torch.arange(d, device=device).unsqueeze(1)
        j = torch.arange(d, device=device).unsqueeze(0)
        _ROT_IDX[key] = (j - i) % d                      # Mpi[i, j] = M[i, (j - i) mod d]  (row i rotated right by i)
    return _ROT_IDX[key]


class LoRMALinear(nn.Module):
    """LoRMA (Bihany, Patel, Modi, ACL Findings 2025): a MULTIPLICATIVE low-rank update of a frozen W0 (out x in).
      variant "plus" (LoRMA+, additive rank inflation):  h = ((alpha/r) B A + I) W0 x + b
            A: (r, out) random, B: (out, r) zeros  -> starts exactly at W0.  Reordered as  y0 = W0 x;  y0 + (alpha/r) B (A y0)
            so the out x out matrix is never formed (same cost order as LoRA).
      variant "pi" (LoRMA_pi, permutation-based inflation):  h = I_pi(B A) W0 x + b, where I_pi rotates row i of BA
            cyclically by i. Init: B[:, 0] = 1, other B entries random; A[0, 0] = 1, other A entries 0 -> I_pi(BA) = I.
            Needs the out x out matrix every step (O(d^2 (r + batch)))."""

    def __init__(self, base, r, alpha, variant="plus"):
        super().__init__()
        self.base = base
        self.variant = variant
        self.r, self.scale = r, alpha / r
        d = base.out_features
        if variant == "plus":
            self.A = nn.Parameter(torch.empty(r, d))
            self.B = nn.Parameter(torch.zeros(d, r))
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        elif variant == "pi":
            A = torch.zeros(r, d); A[0, 0] = 1.0
            B = torch.randn(d, r) * (1.0 / math.sqrt(d)); B[:, 0] = 1.0
            self.A, self.B = nn.Parameter(A), nn.Parameter(B)
        else:
            raise ValueError(variant)

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)
        if self.variant == "plus":
            y = y0 + self.scale * F.linear(F.linear(y0, self.A), self.B)
        else:
            M = self.B @ self.A                                                        # (d, d)
            Mpi = torch.gather(M, 1, _row_rotation_index(M.shape[0], M.device))
            y = F.linear(y0, Mpi.to(y0.dtype))
        if self.base.bias is not None:
            y = y + self.base.bias
        return y


class BottleneckAdapter(nn.Module):
    """Houlsby/Pfeiffer bottleneck: out + up(act(down(out))); up is zero-initialised -> starts as identity."""

    def __init__(self, d, m):
        super().__init__()
        self.down = nn.Linear(d, m)
        self.up = nn.Linear(m, d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))


def _replace_linears(root, name_suffixes, make):
    """Replace every nn.Linear whose qualified name ends with one of name_suffixes by make(linear). Returns names."""
    done = []
    for qual, mod in list(root.named_modules()):
        for child_name, child in list(mod.named_children()):
            full = f"{qual}.{child_name}" if qual else child_name
            if isinstance(child, nn.Linear) and any(full.endswith(s) for s in name_suffixes):
                setattr(mod, child_name, make(child))
                done.append(full)
    return done


def apply_technique(ner, C, log=print):
    """Freezes/unfreezes and inserts modules according to C["TECHNIQUE"]. The head is always trainable.
    Returns a short description dict (for the results file)."""
    tech = C["TECHNIQUE"]
    bb = ner.backbone
    info = dict(technique=tech)
    for p in bb.parameters():
        p.requires_grad = False
    if tech == "full_ft":
        for p in bb.parameters():
            p.requires_grad = True
    elif tech == "partial_ft":
        k = C["PARTIAL_TOP_BLOCKS"]
        blocks = bb.layer_list()
        for blk in blocks[len(blocks) - k:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in bb.final_modules_params():
            p.requires_grad = True
        info.update(top_blocks=k, of_blocks=len(blocks))
    elif tech == "bitfit":
        n = 0
        for name, p in bb.named_parameters():
            if name.split(".")[-1].startswith("bias"):            # Linear/LayerNorm ".bias" and LSTM "bias_ih_l0"/"bias_hh_l0"
                p.requires_grad = True; n += 1
        info.update(bias_tensors=n)
    elif tech == "lora":
        done = _replace_linears(bb, C["PEFT_TARGETS"], lambda lin: LoRALinear(lin, C["RANK"], C["ALPHA"], C.get("LORA_DROPOUT", 0.0)))
        assert done, f"no linear layer matches PEFT_TARGETS={C['PEFT_TARGETS']} in this model"
        info.update(targets=len(done), rank=C["RANK"], alpha=C["ALPHA"])
    elif tech == "lorma":
        done = _replace_linears(bb, C["PEFT_TARGETS"], lambda lin: LoRMALinear(lin, C["RANK"], C["ALPHA"], C["LORMA_VARIANT"]))
        assert done, f"no linear layer matches PEFT_TARGETS={C['PEFT_TARGETS']} in this model"
        info.update(targets=len(done), rank=C["RANK"], alpha=C["ALPHA"], variant=C["LORMA_VARIANT"])
    elif tech == "adapter":
        sites = bb.adapter_sites(C["ADAPTER_TYPE"])            # list of (module whose output gets an adapter, d)
        ner.adapters = nn.ModuleList()
        for mod, d in sites:
            ad = BottleneckAdapter(d, C["ADAPTER_DIM"])
            ner.adapters.append(ad)
            mod.register_forward_hook(lambda m, inp, out, ad=ad: ad(out))
        info.update(adapter_type=C["ADAPTER_TYPE"], sites=len(sites), bottleneck=C["ADAPTER_DIM"])
    elif tech == "prompt":
        P = C["PROMPT_LEN"]
        emb = bb.embedding_weight()
        idx = torch.randint(100, emb.shape[0], (P,), generator=torch.Generator().manual_seed(0))
        ner.soft_prompt = nn.Parameter(emb[idx].detach().clone().float())
        info.update(prompt_len=P)
    elif tech == "prefix":
        P = C["PREFIX_LEN"]
        bb.enable_prefix(P, C.get("PREFIX_HIDDEN", 0))
        info.update(prefix_len=P)
    else:
        raise ValueError(f"unknown TECHNIQUE {tech}")
    for p in ner.head_parameters():
        p.requires_grad = True
    total = sum(p.numel() for p in ner.parameters())
    trainable = sum(p.numel() for p in ner.parameters() if p.requires_grad)
    backbone_orig = bb.original_param_count
    info.update(params_total=total, params_trainable=trainable, params_backbone_original=backbone_orig,
                trainable_pct_of_backbone=round(100.0 * trainable / backbone_orig, 4))
    log(f"Technique {tech}: trainable {trainable:,} of {total:,} parameters ({info['trainable_pct_of_backbone']}% of the Phase-1 model's {backbone_orig:,})")
    return info


def trainable_state(ner):
    """Everything that can differ from the Phase-1 weights: trainable params + routing-bias buffers (if updated)."""
    st = {n: p.detach().to("cpu", copy=True) for n, p in ner.named_parameters() if p.requires_grad}
    for n, b in ner.named_buffers():
        if n.endswith(("select_bias", ".tau", ".tau_ready")):         # MoE routing bias, MoR routing thresholds (updated in training)
            st[n] = b.detach().to("cpu", copy=True)
    return st


def load_trainable_state(ner, st):
    missing = [k for k in st if k not in dict(ner.named_parameters()) and k not in dict(ner.named_buffers())]
    assert not missing, f"state has unknown keys: {missing[:5]}"
    ner.load_state_dict(st, strict=False)


# ----------------------------------------------------------------------------------------------------------------------
# 4. Training (optionally with a knowledge-distillation teacher)
# ----------------------------------------------------------------------------------------------------------------------
def tag_class_weights(docs, n_tags, scheme):
    if scheme == "none":
        return None
    cnt = torch.zeros(n_tags)
    for d in docs.values():
        for t in d["tags"]:
            if t >= 0:
                cnt[t] += 1
    cnt = cnt.clamp(min=1)
    if scheme == "inv_sqrt":
        w = (cnt.sum() / cnt).sqrt()
    elif scheme == "inv":
        w = cnt.sum() / cnt
    else:
        raise ValueError(scheme)
    return w / w[0]                                         # O keeps weight 1


def _lr_lambda(total, warmup):
    def f(step):
        if step < warmup:
            return (step + 1) / max(1, warmup)
        return max(0.0, (total - step) / max(1, total - warmup))
    return f


def train_ner(ner, train_docs, C, device, eval_fn=None, teacher=None, log=print):
    """Trains the trainable parameters. eval_fn(ner) -> (score, details) is called after each epoch (if given); the
    best epoch's trainable state is kept. teacher(ids, mask) -> logits enables KD.
    Returns dict(history, best_epoch, best_score, best_state, train_seconds, peak_mem_gb, tokens_per_sec)."""
    ner.to(device)
    torch.manual_seed(C["SEED"]); random.seed(C["SEED"])
    items = window_items(train_docs, C["WINDOW"], C["TRAIN_STRIDE"])
    n_tags = ner.head[-1].out_features
    cw = tag_class_weights(train_docs, n_tags, C["CLASS_WEIGHTING"])
    cw = cw.to(device) if cw is not None else None
    head_params = [p for p in ner.head_parameters() if p.requires_grad]
    hp = set(id(p) for p in head_params)
    other = [p for p in ner.parameters() if p.requires_grad and id(p) not in hp]
    groups = [dict(params=head_params, lr=C["HEAD_LR"])]
    if other:
        groups.append(dict(params=other, lr=C["LR"]))
    opt = torch.optim.AdamW(groups, weight_decay=C["WEIGHT_DECAY"], betas=(0.9, 0.98))
    steps_per_epoch = math.ceil(len(items) / (C["BATCH"] * C["ACCUM"]))
    total = steps_per_epoch * C["EPOCHS"]
    sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr_lambda(total, int(C["WARMUP_FRAC"] * total)))
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    hist, best, best_state, best_ep, bad_epochs = [], -1.0, None, 0, 0
    t0, tokens = time.time(), 0
    log(f"training: {len(items)} windows/epoch, {steps_per_epoch} optimizer steps/epoch, {C['EPOCHS']} epochs max")
    for ep in range(1, C["EPOCHS"] + 1):
        ner.train()
        rng = random.Random(C["SEED"] * 1000 + ep)
        order = items[:]; rng.shuffle(order)
        run_loss, nb, skipped = 0.0, 0, 0
        opt.zero_grad(set_to_none=True)
        for bi in range(0, len(order), C["BATCH"]):
            ids, mask, labels = collate(train_docs, order[bi:bi + C["BATCH"]], C["WINDOW"])
            ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = ner(ids, mask)
            logits = logits.float()
            loss = F.cross_entropy(logits.view(-1, n_tags), labels.view(-1), weight=cw, ignore_index=IGNORE)
            if teacher is not None and C.get("KD_ALPHA", 0) > 0:
                with torch.no_grad():
                    t_logits = teacher(ids, mask).float()
                keep = labels.view(-1) != IGNORE
                T = C["KD_TEMPERATURE"]
                kd = F.kl_div(F.log_softmax(logits.view(-1, n_tags)[keep] / T, -1),
                              F.softmax(t_logits.view(-1, n_tags)[keep] / T, -1), reduction="batchmean") * T * T
                loss = (1 - C["KD_ALPHA"]) * loss + C["KD_ALPHA"] * kd
            if not torch.isfinite(loss):                      # e.g. an fp16 overflow: skip this batch, never back-propagate NaN/inf
                skipped += 1
                continue
            scaler.scale(loss / C["ACCUM"]).backward()
            nb += 1
            tokens += int(mask.sum())
            run_loss += loss.item()
            if nb % C["ACCUM"] == 0 or bi + C["BATCH"] >= len(order):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], C["GRAD_CLIP"])
                scaler.step(opt); scaler.update(); sched.step()
                opt.zero_grad(set_to_none=True)
        row = dict(epoch=ep, train_loss=round(run_loss / max(1, nb), 5), seconds=round(time.time() - t0, 1))
        if skipped:
            row["skipped_nonfinite_batches"] = skipped
        if eval_fn is not None:
            score, det = eval_fn(ner)
            row.update(heldout_macroF1=round(score, 5), **det)
            if score > best:
                best, best_ep, best_state, bad_epochs = score, ep, trainable_state(ner), 0
            else:
                bad_epochs += 1
        hist.append(row)
        log(f"  epoch {ep}: " + ", ".join(f"{k}={v}" for k, v in row.items() if k != "epoch"))
        if nb == 0:
            log("  every batch of this epoch was non-finite: stopping (the best earlier epoch is kept)")
            break
        if eval_fn is not None and C["PATIENCE"] and bad_epochs >= C["PATIENCE"]:
            log(f"  early stop: no improvement for {bad_epochs} epochs")
            break
    if eval_fn is None:
        best_state, best_ep = trainable_state(ner), len(hist)
    secs = time.time() - t0
    peak = torch.cuda.max_memory_allocated(device) / 1e9 if device.type == "cuda" else None
    return dict(history=hist, best_epoch=best_ep, best_score=best, best_state=best_state, train_seconds=round(secs, 1),
                peak_mem_gb=round(peak, 3) if peak else None, tokens_per_sec=round(tokens / max(secs, 1e-9), 1))


# ----------------------------------------------------------------------------------------------------------------------
# 5. Inference: windows -> per-token logits (merged) -> tags -> character spans
# ----------------------------------------------------------------------------------------------------------------------
@torch.no_grad()
def doc_logits(ner, doc, C, device, causal):
    n = len(doc["ids"])
    n_tags = ner.head[-1].out_features
    if n == 0:
        return torch.zeros(0, n_tags)
    W, S = C["WINDOW"], C["INFER_STRIDE"]
    starts = make_windows(n, W, S)
    out = torch.zeros(n, n_tags)
    best = torch.full((n,), -1.0)
    cnt = torch.zeros(n)
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"
    ner.eval()
    for bi in range(0, len(starts), C["INFER_BATCH"]):
        chunk = [("_", s) for s in starts[bi:bi + C["INFER_BATCH"]]]
        ids, mask, _ = collate({"_": doc}, chunk, W, with_labels=False)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            lg = ner(ids.to(device), mask.to(device)).float().cpu()
        for b, (_, s) in enumerate(chunk):
            L = int(mask[b].sum())
            pos = torch.arange(L)
            if C["MERGE"] == "average":
                out[s:s + L] += lg[b, :L]; cnt[s:s + L] += 1
            else:                                                # "context": keep the window giving each token most context
                score = pos.float() if causal else torch.minimum(pos, L - 1 - pos).float()
                upd = score > best[s:s + L]
                out[s:s + L][upd] = lg[b, :L][upd]
                best[s:s + L] = torch.where(upd, score, best[s:s + L])
    if C["MERGE"] == "average":
        out = out / cnt.clamp(min=1).unsqueeze(1)
    return out


def bio_allowed(n_tags):
    """allowed[prev, cur]: I-X may only follow B-X or I-X."""
    a = torch.ones(n_tags, n_tags, dtype=torch.bool)
    for cur in range(1, n_tags):
        if (cur - 1) % 2 == 1:                                    # I- tag
            b_same, i_same = cur - 1, cur
            for prev in range(n_tags):
                if prev not in (b_same, i_same):
                    a[prev, cur] = False
    return a


def viterbi_bio(logits):
    """Best tag sequence under BIO constraints (no transition scores, only the hard constraint)."""
    n, K = logits.shape
    if n == 0:
        return []
    lp = F.log_softmax(logits.float(), -1)
    allowed = bio_allowed(K)
    trans = torch.where(allowed, torch.tensor(0.0), torch.tensor(-1e9))
    start_ok = torch.tensor([0.0] + [0.0 if (k - 1) % 2 == 0 else -1e9 for k in range(1, K)])
    score = lp[0] + start_ok
    back = torch.zeros(n, K, dtype=torch.long)
    for t in range(1, n):
        cand = score.unsqueeze(1) + trans                         # (prev, cur)
        best, arg = cand.max(0)
        back[t] = arg
        score = best + lp[t]
    path = [int(score.argmax())]
    for t in range(n - 1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return path[::-1]


def decode_doc(doc, logits, labels, C, trail=None, lead=None):
    if len(doc["ids"]) == 0:
        return []
    if C["DECODE"] == "viterbi":
        tags = viterbi_bio(logits)
    else:
        tags = logits.argmax(-1).tolist()
    spans = doc["spans"]
    tags = [IGNORE if e <= s else t for (s, e), t in zip(spans, tags)]
    raw = token_tags_to_spans(spans, tags, labels)
    if C["BOUNDARY_REPAIR"]:
        return finalize_doc_entities(doc["text"], raw, trail, lead)
    return finalize_doc_entities(doc["text"], raw)


def predict_docs(ner, docs, labels, C, device, causal, trail=None, lead=None):
    t0 = time.time()
    preds = {did: decode_doc(d, doc_logits(ner, d, C, device, causal), labels, C, trail, lead) for did, d in docs.items()}
    return preds, time.time() - t0


def confusion_pairs(golds, preds):
    """(gold label -> predicted label) counts for predictions that overlap a gold entity of a different label."""
    from collections import Counter
    c = Counter()
    for did, g in golds.items():
        for p in preds.get(did, []):
            for x in g:
                if x[0] < p[1] and p[0] < x[1] and x[2] != p[2]:
                    c[f"{x[2]}->{p[2]}"] += 1
    return dict(c.most_common(25))


def detailed_analysis(golds, preds, texts, n_tokens=None):
    """Error analysis for the report: (1) what kind of boundary errors (too long / too short / shifted; differing only by
    punctuation/space; the most common differing strings), (2) where in the document errors happen (header region vs
    body), (3) document length (short / medium / long thirds by characters)."""
    from collections import Counter
    kinds, diffs = Counter(), Counter()
    for did, g in golds.items():
        gl = [tuple(x[:3]) for x in g]
        gs = set(gl)
        t = texts[did]
        for (s, e, l) in {tuple(x[:3]) for x in preds.get(did, [])}:
            if (s, e, l) in gs:
                continue
            ov = [x for x in gl if x[2] == l and x[0] < e and s < x[1]]
            if not ov:
                continue
            x = min(ov, key=lambda x: abs(x[0] - s) + abs(x[1] - e))
            if s <= x[0] and e >= x[1]:
                kinds["pred_too_long"] += 1
                extra = (t[s:x[0]] + " | " + t[x[1]:e]).strip()
            elif s >= x[0] and e <= x[1]:
                kinds["pred_too_short"] += 1
                extra = (t[x[0]:s] + " | " + t[e:x[1]]).strip()
            else:
                kinds["shifted"] += 1
                extra = None
            if extra is not None:
                if not any(c.isalnum() for c in extra):
                    kinds["differs_only_by_punct_or_space"] += 1
                diffs[extra[:40]] += 1
    out = dict(boundary_error_kinds=dict(kinds), boundary_diff_top=diffs.most_common(15))

    def region_of(s, n):
        r = s / max(n, 1)
        return "first_5pct(header)" if r < 0.05 else ("5-50pct" if r < 0.5 else "50-100pct")
    reg = {k: Counter() for k in ("first_5pct(header)", "5-50pct", "50-100pct")}
    for did, g in golds.items():
        n = len(texts[did])
        gs = {tuple(x[:3]) for x in g}
        ps = {tuple(x[:3]) for x in preds.get(did, [])}
        for x in gs:
            reg[region_of(x[0], n)]["gold"] += 1
            reg[region_of(x[0], n)]["tp"] += x in ps
        for x in ps:
            reg[region_of(x[0], n)]["pred"] += 1
            if not any(y[0] < x[1] and x[0] < y[1] for y in gs):
                reg[region_of(x[0], n)]["spurious"] += 1
    out["by_region"] = {k: dict(gold=v["gold"], pred=v["pred"], tp=v["tp"], spurious=v["spurious"],
                                recall=round(v["tp"] / max(v["gold"], 1), 4), precision=round(v["tp"] / max(v["pred"], 1), 4))
                        for k, v in reg.items()}
    ids = sorted(golds, key=lambda d: len(texts[d]))
    k = max(1, len(ids) // 3)
    groups = {"short_third": ids[:k], "middle_third": ids[k:2 * k], "long_third": ids[2 * k:]}
    out["by_doc_length"] = {}
    for name, ds in groups.items():
        if not ds:
            continue
        tp = sum(len({tuple(x[:3]) for x in golds[d]} & {tuple(x[:3]) for x in preds.get(d, [])}) for d in ds)
        ng = sum(len(golds[d]) for d in ds)
        npd = sum(len(preds.get(d, [])) for d in ds)
        p, r = tp / max(npd, 1), tp / max(ng, 1)
        out["by_doc_length"][name] = dict(docs=len(ds), chars_min=len(texts[ds[0]]), chars_max=len(texts[ds[-1]]),
                                          micro_P=round(p, 4), micro_R=round(r, 4), micro_F1=round(2 * p * r / max(p + r, 1e-9), 4))
    return out


EVALUATOR_SRC = '#!/usr/bin/env python3\n"""Local evaluator for Phase 2.\n\nUsage:\n  python evaluate_phase2.py --truth truth.json --prediction prediction.json --labels labels.txt\n"""\nimport argparse, json, sys\n\nclass DuplicateJSONKeyError(ValueError): pass\ndef reject_constant(v): raise ValueError(f"Non-standard JSON numeric value \'{v}\' is not allowed.")\ndef pairs_no_duplicates(pairs):\n    out={}; dup=[]\n    for k,v in pairs:\n        if k in out: dup.append(k)\n        out[k]=v\n    if dup: raise DuplicateJSONKeyError(f"Duplicate JSON object keys detected: {list(dict.fromkeys(dup))[:10]}.")\n    return out\ndef load_json(path):\n    with open(path,encoding=\'utf-8\') as f:\n        try: return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)\n        except json.JSONDecodeError as e: raise ValueError(f"Invalid JSON in {path} at line {e.lineno}, column {e.colno}: {e.msg}") from e\ndef labels(path):\n    with open(path,encoding=\'utf-8\') as f: xs=[x.strip() for x in f if x.strip()]\n    if not xs: raise ValueError(\'labels.txt is empty.\')\n    if len(xs)!=12: raise ValueError(f\'Expected 12 labels, found {len(xs)}.\')\n    if len(set(xs))!=len(xs): raise ValueError(\'labels.txt contains duplicate labels.\')\n    return xs\ndef spans(x, allowed, where):\n    if x is None: return set()\n    if not isinstance(x,list): raise ValueError(f\'{where}: expected a list of entities.\')\n    out=set()\n    for j,s in enumerate(x):\n        if not isinstance(s,list) or len(s) not in (3,4): raise ValueError(f\'{where}, entity {j}: expected [start,end,label] or [start,end,label,entity_text].\')\n        a,b,l=s[:3]\n        if isinstance(a,bool) or not isinstance(a,int) or isinstance(b,bool) or not isinstance(b,int): raise ValueError(f\'{where}, entity {j}: start/end must be integers.\')\n        if a<0 or b<=a: raise ValueError(f\'{where}, entity {j}: require 0 <= start < end.\')\n        if not isinstance(l,str) or l not in allowed: raise ValueError(f\'{where}, entity {j}: invalid label {l!r}.\')\n        if len(s)==4 and not isinstance(s[3],str): raise ValueError(f\'{where}, entity {j}: entity_text must be a string.\')\n        t=(a,b,l)\n        if t in out: raise ValueError(f\'{where}, entity {j}: duplicate span {t}.\')\n        out.add(t)\n    return out\n\ndef main():\n    ap=argparse.ArgumentParser(description=\'Evaluate Phase 2 entity predictions on your own held-out split.\')\n    ap.add_argument(\'--truth\',required=True); ap.add_argument(\'--prediction\',required=True); ap.add_argument(\'--labels\',required=True)\n    args=ap.parse_args(); truth=load_json(args.truth); pred=load_json(args.prediction); labs=labels(args.labels)\n    if not isinstance(truth,dict) or not truth or not isinstance(pred,dict) or not pred: raise ValueError(\'Truth and prediction must be non-empty JSON objects.\')\n    if set(truth)!=set(pred): raise ValueError(f"ID mismatch: missing={len(set(truth)-set(pred))}, extra={len(set(pred)-set(truth))}")\n    stats={l:[0,0,0] for l in labs}; allowed=set(labs)\n    for did in truth:\n        g=spans(truth[did],allowed,f\'truth[{did}]\'); p=spans(pred[did],allowed,f\'prediction[{did}]\')\n        for l in labs:\n            gs={x for x in g if x[2]==l}; ps={x for x in p if x[2]==l}\n            stats[l][0]+=len(gs&ps); stats[l][1]+=len(ps-gs); stats[l][2]+=len(gs-ps)\n    precisions=[]; recalls=[]; f1s=[]\n    print(f\'Examples: {len(truth)}\')\n    for l in labs:\n        tp,fp,fn=stats[l]; pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0; f=2*pr*rc/(pr+rc) if pr+rc else 0\n        precisions.append(pr); recalls.append(rc); f1s.append(f)\n        print(f\'{l}: P={pr:.6f} R={rc:.6f} F1={f:.6f}\')\n    print(f\'Strict Macro-F1:        {sum(f1s)/len(f1s):.6f}\')\n    print(f\'Strict Macro-Precision: {sum(precisions)/len(precisions):.6f}\')\n    print(f\'Strict Macro-Recall:    {sum(recalls)/len(recalls):.6f}\')\nif __name__==\'__main__\':\n    try: main()\n    except Exception as e: print(f\'EVALUATION FAILED: {e}\',file=sys.stderr); sys.exit(1)\n'

## 9. Tokenize the documents and learn the boundary-repair tables (one per training fold)

In [9]:
t0 = time.time()
TAG_NAMES = bio_tag_names(LABELS)
DOCS = {f: prepare_docs(bpe_tokenizer, TEXTS[f], GOLDS[f], LABELS, INPUT_MODE) for f in TEXTS}
TEST_DOCS = prepare_docs(bpe_tokenizer, TEST_TEXTS, None, LABELS, INPUT_MODE)
if INPUT_MODE == "none":
    check_encode_identical(bpe_tokenizer, {**TEXTS["fold1"], **TEXTS["fold2"], **TEST_TEXTS})
    print("token ids identical to BPETokenizer.encode on every document")
GOLD3 = {f: {d: [e[:3] for e in GOLDS[f][d]] for d in GOLDS[f]} for f in GOLDS}


def trim_tables(folds):
    tx, gd, sp = {}, {}, {}
    for f in folds:
        tx.update(TEXTS[f]); gd.update(GOLDS[f]); sp.update({d: DOCS[f][d]["spans"] for d in DOCS[f]})
    return learn_trim_table(tx, gd, sp, LABELS)


TRIM = {"fold1": trim_tables(["fold1"]), "fold2": trim_tables(["fold2"]), "both": trim_tables(["fold1", "fold2"])}
mis = sum(len(gold_to_token_tags(DOCS[f][d]["spans"], GOLDS[f][d], LABELS)[1]) for f in DOCS for d in DOCS[f])
print(f"{len(TAG_NAMES)} tags; {mis} gold entities not on token boundaries; {time.time()-t0:.0f}s")


token ids identical to BPETokenizer.encode on every document
25 tags; 2828 gold entities not on token boundaries; 17s


## 10. Load the Phase-1 checkpoint and check it

In [10]:
t0 = time.time()
_ck = torch.load(fetch_model_file(PHASE1_CHECKPOINT), map_location="cpu", mmap=True)
PHASE1_STATE = _ck["model"] if "model" in _ck else _ck
del _ck
_lm = build_phase1_lm()
PHASE1_PARAMS = sum(p.numel() for p in _lm.parameters())
SANITY_LOSS = phase1_sanity(_lm, DOCS["fold1"])
print(f"Phase-1 model {MODEL_KEY}: {PHASE1_PARAMS:,} parameters; next-token loss on NER documents = {SANITY_LOSS:.3f} nats "
      f"(far below {math.log(VOCAB):.1f} = trained weights loaded; about {math.log(VOCAB):.1f} = random weights) [{time.time()-t0:.0f}s]")
assert SANITY_LOSS < 8.0, "the Phase-1 weights do not seem to be loaded correctly"
del _lm; gc.collect(); torch.cuda.empty_cache() if device.type == "cuda" else None


def build_ner(C):
    lm = build_phase1_lm()
    bb = make_backbone(lm, C)
    ner = NERModel(bb, bb.d_model, len(TAG_NAMES), C["HEAD_TYPE"], C["HEAD_DROPOUT"], C["HEAD_NORM"], C["HEAD_LSTM_HIDDEN"])
    info = apply_technique(ner, C)
    return ner.to(device), info


def is_causal(C):
    return C["ATTENTION"] != "bidirectional" and C["HEAD_TYPE"] != "bilstm"


def free(*objs):
    for o in objs:
        del o
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()


Phase-1 model mor: 120,842,946 parameters; next-token loss on NER documents = 5.217 nats (far below 10.6 = trained weights loaded; about 10.6 = random weights) [8s]


## 11. Stage 1: every setup, fold1 → fold2

In [11]:
# ---- one setup, one direction: train on `tr`, choose the epoch and report on `ev` --------------------------------
OTHER = {"fold1": "fold2", "fold2": "fold1"}
os.makedirs(f"{WORK}/sweep", exist_ok=True)


def result_name(name, tr):
    return f"{SWEEP_NAME}__{name}__{tr}to{OTHER[tr]}.json"


def load_cached_result(name, tr):
    fn = result_name(name, tr)
    for path in (f"{WORK}/sweep/{fn}",):
        if os.path.exists(path):
            return json.load(open(path))
    try:
        p = fetch_model_file(fn, f"{WORK}/sweep/{fn}")
        print(f"  (found finished result {fn} in storage -- not re-running it)")
        return json.load(open(p))
    except Exception:
        return None


def print_setup_report(res):
    """Everything the write-up needs about one run, printed so the notebook output alone is enough."""
    print(f"  [{res['setup']} {res['train_fold']}->{res['eval_fold']}] settings: " + json.dumps(res["overrides"], default=str))
    print("  epochs: " + " | ".join(f"{h['epoch']}: loss {h['train_loss']}, F1 {h.get('heldout_macroF1')}" for h in res["history"]))
    print(f"  {'label':10s} {'P':>6s} {'R':>6s} {'F1':>6s} {'tp':>5s} {'fp':>5s} {'fn':>5s}")
    for l, d in res["per_label"].items():
        print(f"  {l:10s} {d['P']:6.3f} {d['R']:6.3f} {d['F1']:6.3f} {d['tp']:5d} {d['fp']:5d} {d['fn']:5d}")
    print("  errors:", res["errors"])
    print("  top label confusions:", dict(list(res["label_confusions"].items())[:10]))
    a = res["analysis"]
    print("  boundary error kinds:", a["boundary_error_kinds"])
    print("  most common boundary differences (extra/missing text):", a["boundary_diff_top"][:10])
    print("  by region of the document:", a["by_region"])
    print("  by document length:", a["by_doc_length"])
    ti = res["technique_info"]
    print(f"  params: trainable {ti['params_trainable']:,} / total {ti['params_total']:,} ({ti['trainable_pct_of_backbone']}% of Phase-1 "
          f"{res['phase1_params']:,}) | train {res['train_seconds']}s, {res['tokens_per_sec']} tok/s, peak {res['peak_mem_gb']} GB | "
          f"inference {res['inference_seconds']}s, {res['inference_tokens_per_sec']} tok/s")


def run_setup(name, overrides, tr):
    cached = load_cached_result(name, tr)
    if cached is not None:
        return cached
    ev = OTHER[tr]
    C = dict(BASE, **overrides)
    print(f"\n=== {name}  [{tr} -> {ev}]  " + ", ".join(f"{k}={v}" for k, v in overrides.items()))
    t0 = time.time()
    ner, info = build_ner(C)
    causal = is_causal(C)

    def eval_fn(model):
        preds, _ = predict_docs(model, DOCS[ev], LABELS, C, device, causal, *TRIM[tr])
        s = strict_scores(GOLD3[ev], preds, LABELS)
        return s["macro_F1"], dict(heldout_P=round(s["macro_P"], 4), heldout_R=round(s["macro_R"], 4))

    cv = train_ner(ner, DOCS[tr], C, device, eval_fn=eval_fn)
    load_trainable_state(ner, cv["best_state"])
    preds, infer_s = predict_docs(ner, DOCS[ev], LABELS, C, device, causal, *TRIM[tr])
    S = strict_scores(GOLD3[ev], preds, LABELS)
    n_tok = sum(len(d["ids"]) for d in DOCS[ev].values())
    res = dict(sweep=SWEEP_NAME, setup=name, model=MODEL_KEY, train_fold=tr, eval_fold=ev, overrides=overrides,
               settings={k: v for k, v in C.items()}, technique_info=info, phase1_params=PHASE1_PARAMS,
               best_epoch=cv["best_epoch"], macro_F1=round(S["macro_F1"], 4), macro_P=round(S["macro_P"], 4),
               macro_R=round(S["macro_R"], 4),
               per_label_F1={l: round(d["F1"], 4) for l, d in S["per_label"].items()},
               per_label={l: {k: (round(v, 4) if isinstance(v, float) else v) for k, v in d.items()} for l, d in S["per_label"].items()},
               errors=error_breakdown(GOLD3[ev], preds), label_confusions=confusion_pairs(GOLD3[ev], preds),
               history=cv["history"], train_seconds=cv["train_seconds"], peak_mem_gb=cv["peak_mem_gb"],
               tokens_per_sec=cv["tokens_per_sec"], inference_seconds=round(infer_s, 1),
               inference_tokens_per_sec=round(n_tok / max(infer_s, 1e-9), 1), wall_seconds=round(time.time() - t0, 1),
               analysis=detailed_analysis(GOLD3[ev], preds, TEXTS[ev]))
    pf = f"{WORK}/sweep/{SWEEP_NAME}__{name}__{tr}to{ev}_pred.json"
    json.dump(preds, open(pf, "w"))
    upload_model_file(pf)
    fn = result_name(name, tr)
    json.dump(res, open(f"{WORK}/sweep/{fn}", "w"), indent=1, default=str)
    upload_model_file(f"{WORK}/sweep/{fn}")
    print(f"  -> {name} [{tr}->{ev}] strict macro-F1 {res['macro_F1']} (P {res['macro_P']}, R {res['macro_R']}), best epoch "
          f"{res['best_epoch']}, trainable {info['params_trainable']:,} ({info['trainable_pct_of_backbone']}%), "
          f"peak {res['peak_mem_gb']} GB, {res['wall_seconds']:.0f}s")
    print_setup_report(res)
    free(ner)
    return res


# ---- stage 1: every setup, fold1 -> fold2 -------------------------------------------------------------------------
STAGE1 = {}
for name, overrides in SWEEP:
    if MAX_SWEEP_HOURS is not None and (time.time() - NB_START) / 3600 > MAX_SWEEP_HOURS:
        print(f"MAX_SWEEP_HOURS reached: skipping {name} and the rest")
        break
    STAGE1[name] = run_setup(name, overrides, "fold1")



=== full_ft_bilstm  [fold1 -> fold2]  TECHNIQUE=full_ft, HEAD_TYPE=bilstm, LR=3e-05, EPOCHS=10, PATIENCE=3
Technique full_ft: trainable 92,197,019 of 92,197,019 parameters (76.2949% of the Phase-1 model's 120,842,946)
training: 182 windows/epoch, 23 optimizer steps/epoch, 10 epochs max
  epoch 1: train_loss=1.99168, seconds=27.3, heldout_macroF1=0.06371, heldout_P=0.0746, heldout_R=0.0597
  epoch 2: train_loss=0.97419, seconds=69.0, heldout_macroF1=0.13951, heldout_P=0.1107, heldout_R=0.196
  epoch 3: train_loss=0.58021, seconds=111.0, heldout_macroF1=0.19174, heldout_P=0.1657, heldout_R=0.2495
  epoch 4: train_loss=0.39307, seconds=152.6, heldout_macroF1=0.20674, heldout_P=0.1811, heldout_R=0.2812
  epoch 5: train_loss=0.26717, seconds=194.3, heldout_macroF1=0.2193, heldout_P=0.186, heldout_R=0.2825
  epoch 6: train_loss=0.19567, seconds=236.0, heldout_macroF1=0.24309, heldout_P=0.2176, heldout_R=0.2889
  epoch 7: train_loss=0.14751, seconds=277.6, heldout_macroF1=0.2623, heldout_P=0

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded p2s_mor2__full_ft_bilstm__fold1tofold2_pred.json to bucket kavincg23/cs781-models


Processing Files (0 / 0): |          |  0.00B /  0.00B            

... (89 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded p2s_mor2__frozen_bilstm__fold1tofold2.json to bucket kavincg23/cs781-models
  -> frozen_bilstm [fold1->fold2] strict macro-F1 0.3173 (P 0.3288, R 0.3213), best epoch 20, trainable 2,114,073 (1.7494%), peak 0.674 GB, 626s
  [frozen_bilstm fold1->fold2] settings: {"TECHNIQUE": "partial_ft", "PARTIAL_TOP_BLOCKS": 0, "HEAD_TYPE": "bilstm", "LR": 0.001, "EPOCHS": 20, "PATIENCE": 5}
  epochs: 1: loss 2.26102, F1 0.07011 | 2: loss 1.05692, F1 0.1424 | 3: loss 0.57454, F1 0.18942 | 4: loss 0.37512, F1 0.21194 | 5: loss 0.25211, F1 0.23292 | 6: loss 0.18379, F1 0.25537 | 7: loss 0.13672, F1 0.2624 | 8: loss 0.10342, F1 0.27214 | 9: loss 0.08159, F1 0.27828 | 10: loss 0.0657, F1 0.29179 | 11: loss 0.05345, F1 0.30139 | 12: loss 0.04368, F1 0.3068 | 13: loss 0.036, F1 0.30721 | 14: loss 0.03054, F1 0.31279 | 15: loss 0.02655, F1 0.31289 | 16: loss 0.02289, F1 0.31583 | 17: loss 0.02065, F1 0.31362 | 18: loss 0.01854, F1 0.31463 | 19: loss 0.0172, F1 0.3165 | 20: loss 0.01636, F1 0.31731


## 12. Stage 2: re-check the best setups fold2 → fold1, and the ranking table

In [12]:
# ---- stage 2: re-check the best CONFIRM_TOP_M setups the other way round (fold2 -> fold1) -------------------------
order = sorted(STAGE1, key=lambda n: -STAGE1[n]["macro_F1"])
STAGE2 = {}
for name in order[:CONFIRM_TOP_M]:
    STAGE2[name] = run_setup(name, dict(SWEEP)[name], "fold2")

RANK = []
for name in order:
    f2 = STAGE1[name]["macro_F1"]
    f1 = STAGE2[name]["macro_F1"] if name in STAGE2 else None
    mean = (f1 + f2) / 2 if f1 is not None else None
    RANK.append(dict(setup=name, fold2_F1=f2, fold1_F1=f1, mean_F1=mean,
                     best_epochs=[STAGE1[name]["best_epoch"]] + ([STAGE2[name]["best_epoch"]] if name in STAGE2 else [])))
# confirmed setups are ranked by the two-fold mean and come first; the rest follow by fold2 F1
RANK.sort(key=lambda r: (r["mean_F1"] is None, -(r["mean_F1"] if r["mean_F1"] is not None else r["fold2_F1"])))

print(f"\n{'rank':4s} {'setup':22s} {'fold2 F1':>9s} {'fold1 F1':>9s} {'mean':>7s} {'trainable':>12s} {'peak GB':>8s} {'train s':>8s}")
for i, r in enumerate(RANK, 1):
    s = STAGE1[r["setup"]]
    print(f"{i:<4d} {r['setup']:22s} {r['fold2_F1']:9.4f} {('%.4f' % r['fold1_F1']) if r['fold1_F1'] is not None else '-':>9s} "
          f"{('%.4f' % r['mean_F1']) if r['mean_F1'] is not None else '-':>7s} {s['technique_info']['params_trainable']:>12,} "
          f"{str(s['peak_mem_gb']):>8s} {s['train_seconds']:>8.0f}")
print("\nper-label fold2 F1:")
for r in RANK:
    print(f"  {r['setup']:22s}", {l: v for l, v in STAGE1[r['setup']]['per_label_F1'].items()})



=== lorma_pi_bilstm  [fold2 -> fold1]  TECHNIQUE=lorma, LORMA_VARIANT=pi, HEAD_TYPE=bilstm, LR=0.0003, EPOCHS=20, PATIENCE=5
Technique lorma: trainable 2,421,273 of 92,504,219 parameters (2.0037% of the Phase-1 model's 120,842,946)
training: 186 windows/epoch, 24 optimizer steps/epoch, 20 epochs max
  epoch 1: train_loss=2.20001, seconds=28.1, heldout_macroF1=0.04594, heldout_P=0.057, heldout_R=0.0399
  epoch 2: train_loss=1.1826, seconds=71.7, heldout_macroF1=0.14817, heldout_P=0.134, heldout_R=0.178
  epoch 3: train_loss=0.76148, seconds=115.6, heldout_macroF1=0.17709, heldout_P=0.1529, heldout_R=0.2648
  epoch 4: train_loss=0.53538, seconds=159.6, heldout_macroF1=0.22284, heldout_P=0.2001, heldout_R=0.2645
  epoch 5: train_loss=0.3717, seconds=203.4, heldout_macroF1=0.25082, heldout_P=0.2198, heldout_R=0.2967
  epoch 6: train_loss=0.26561, seconds=247.2, heldout_macroF1=0.27092, heldout_P=0.2556, heldout_R=0.308
  epoch 7: train_loss=0.18982, seconds=291.4, heldout_macroF1=0.26573,

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded p2s_mor2__lorma_pi_bilstm__fold2tofold1_pred.json to bucket kavincg23/cs781-models


Processing Files (0 / 0): |          |  0.00B /  0.00B            

... (31 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded p2s_mor2__adapter_pfeiffer_bilstm__fold2tofold1.json to bucket kavincg23/cs781-models
  -> adapter_pfeiffer_bilstm [fold2->fold1] strict macro-F1 0.3317 (P 0.3566, R 0.3206), best epoch 18, trainable 3,204,569 (2.6518%), peak 1.378 GB, 831s
  [adapter_pfeiffer_bilstm fold2->fold1] settings: {"TECHNIQUE": "adapter", "ADAPTER_TYPE": "pfeiffer", "HEAD_TYPE": "bilstm", "LR": 0.0003, "EPOCHS": 20, "PATIENCE": 5}
  epochs: 1: loss 2.23096, F1 0.05809 | 2: loss 1.03718, F1 0.17333 | 3: loss 0.53835, F1 0.20636 | 4: loss 0.33139, F1 0.23537 | 5: loss 0.21612, F1 0.26211 | 6: loss 0.15072, F1 0.272 | 7: loss 0.11002, F1 0.29291 | 8: loss 0.07993, F1 0.30324 | 9: loss 0.06249, F1 0.30863 | 10: loss 0.04771, F1 0.32171 | 11: loss 0.03748, F1 0.30823 | 12: loss 0.03125, F1 0.32194 | 13: loss 0.02486, F1 0.32445 | 14: loss 0.02092, F1 0.32622 | 15: loss 0.01757, F1 0.32921 | 16: loss 0.01527, F1 0.3279 | 17: loss 0.01327, F1 0.32615 | 18: loss 0.01221, F1 0.33166 | 19: loss 0.01114, F1 0.3

## 13. Stage 3: retrain the top K on both folds, write submission zips

In [13]:
# ---- stage 3: retrain the top K on fold1+fold2 and write their submission zips -----------------------------------
def validate_prediction(pred, texts, labels):
    assert set(pred) == set(texts), f"id mismatch: missing {len(set(texts)-set(pred))}, extra {len(set(pred)-set(texts))}"
    for did, ents in pred.items():
        seen = set()
        for x in ents:
            s, e, l = x
            assert type(s) is int and type(e) is int and 0 <= s < e <= len(texts[did]) and l in labels, (did, x)
            assert (s, e, l) not in seen, ("duplicate", did, x); seen.add((s, e, l))


both = {**DOCS["fold1"], **DOCS["fold2"]}
FINAL = []
for r in RANK[:TOP_K_SUBMIT]:
    name = r["setup"]
    C = dict(BASE, **dict(SWEEP)[name])
    epochs = max(1, round(sum(r["best_epochs"]) / len(r["best_epochs"])))
    print(f"\n=== refit {name} on fold1+fold2 for {epochs} epochs")
    ner, info = build_ner(C)
    fit = train_ner(ner, both, dict(C, EPOCHS=epochs, PATIENCE=0), device, eval_fn=None)
    pred, test_s = predict_docs(ner, TEST_DOCS, LABELS, C, device, is_causal(C), *TRIM["both"])
    validate_prediction(pred, TEST_TEXTS, LABELS)
    tag = f"{SWEEP_NAME}_{name}"
    with open(f"{WORK}/prediction.json", "w", encoding="utf-8") as f:
        json.dump(pred, f)
    zp = f"{WORK}/submission_{tag}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{WORK}/prediction.json", arcname="prediction.json")
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"]
    torch.save(fit["best_state"], f"{WORK}/{tag}_final_state.pt")
    upload_model_file(f"{WORK}/{tag}_final_state.pt")
    FINAL.append(dict(setup=name, zip=os.path.basename(zp), refit_epochs=epochs, refit_seconds=fit["train_seconds"],
                      test_entities=sum(len(v) for v in pred.values()), test_inference_seconds=round(test_s, 1)))
    print(f"  wrote {zp}: {FINAL[-1]['test_entities']} entities")
    del ner, fit
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()



=== refit lorma_pi_bilstm on fold1+fold2 for 19 epochs
Technique lorma: trainable 2,421,273 of 92,504,219 parameters (2.0037% of the Phase-1 model's 120,842,946)
training: 368 windows/epoch, 46 optimizer steps/epoch, 19 epochs max
  epoch 1: train_loss=1.87289, seconds=55.3
  epoch 2: train_loss=0.80109, seconds=111.0
  epoch 3: train_loss=0.47075, seconds=166.8
  epoch 4: train_loss=0.33923, seconds=222.5
  epoch 5: train_loss=0.25324, seconds=278.2
  epoch 6: train_loss=0.17045, seconds=333.7
  epoch 7: train_loss=0.11696, seconds=389.4
  epoch 8: train_loss=0.07954, seconds=444.9
  epoch 9: train_loss=0.05871, seconds=500.6
  epoch 10: train_loss=0.04786, seconds=556.4
  epoch 11: train_loss=0.03467, seconds=612.0
  epoch 12: train_loss=0.02585, seconds=667.5
  epoch 13: train_loss=0.01936, seconds=723.0
  epoch 14: train_loss=0.01431, seconds=778.5
  epoch 15: train_loss=0.0113, seconds=834.0
  epoch 16: train_loss=0.00908, seconds=889.6
  epoch 17: train_loss=0.00714, seconds=945

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded p2s_mor2_lorma_pi_bilstm_final_state.pt to bucket kavincg23/cs781-models
  wrote /kaggle/working/submission_p2s_mor2_lorma_pi_bilstm.zip: 3946 entities

=== refit lorma_bilstm on fold1+fold2 for 16 epochs
Technique lorma: trainable 2,421,273 of 92,504,219 parameters (2.0037% of the Phase-1 model's 120,842,946)
training: 368 windows/epoch, 46 optimizer steps/epoch, 16 epochs max
  epoch 1: train_loss=1.82253, seconds=54.8
  epoch 2: train_loss=0.62077, seconds=109.3
  epoch 3: train_loss=0.33906, seconds=163.7
  epoch 4: train_loss=0.21367, seconds=218.2
  epoch 5: train_loss=0.1441, seconds=272.3
  epoch 6: train_loss=0.10126, seconds=325.9
  epoch 7: train_loss=0.07368, seconds=379.3
  epoch 8: train_loss=0.05336, seconds=432.8
  epoch 9: train_loss=0.04012, seconds=485.8
  epoch 10: train_loss=0.03084, seconds=539.0
  epoch 11: train_loss=0.02361, seconds=592.0
  epoch 12: train_loss=0.01932, seconds=645.1
  epoch 13: train_loss=0.01524, seconds=698.2
  epoch 14: train_loss=

Processing Files (0 / 0): |          |  0.00B /  0.00B            

... (7 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded p2s_mor2_lora_bilstm_final_state.pt to bucket kavincg23/cs781-models
  wrote /kaggle/working/submission_p2s_mor2_lora_bilstm.zip: 3889 entities


## 14. Save the summary to the bucket

In [14]:
SUMMARY = dict(sweep=SWEEP_NAME, model=MODEL_KEY, phase1_checkpoint=PHASE1_CHECKPOINT, phase1_params=PHASE1_PARAMS,
               sanity_lm_loss=round(SANITY_LOSS, 4), base=BASE, ranking=RANK, final=FINAL,
               stage1={k: {kk: v[kk] for kk in ("macro_F1", "macro_P", "macro_R", "best_epoch", "per_label_F1", "errors",
                                                 "train_seconds", "peak_mem_gb", "technique_info")} for k, v in STAGE1.items()},
               stage2={k: {kk: v[kk] for kk in ("macro_F1", "best_epoch", "per_label_F1")} for k, v in STAGE2.items()},
               notebook_minutes=round((time.time() - NB_START) / 60, 1),
               gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu")
sp = f"{WORK}/{SWEEP_NAME}_summary.json"
json.dump(SUMMARY, open(sp, "w"), indent=1, default=str)
upload_model_file(sp)
print("=" * 100)
print(f"SWEEP {SWEEP_NAME} ({MODEL_KEY}, {PHASE1_PARAMS:,} params, sanity loss {SANITY_LOSS:.3f}) -- {SUMMARY['notebook_minutes']} min")
for i, r in enumerate(RANK, 1):
    print(f"  {i}. {r['setup']}: fold2 {r['fold2_F1']}" + (f", fold1 {r['fold1_F1']}, mean {round(r['mean_F1'], 4)}" if r['mean_F1'] is not None else ""))
print("Submission zips (top", TOP_K_SUBMIT, "):", [f["zip"] for f in FINAL])
print("\nFull table (fold1->fold2):")
print(f"{'setup':22s} {'F1':>6s} {'P':>6s} {'R':>6s} {'ep':>3s} {'trainable':>12s} {'%':>8s} {'GB':>6s} {'train s':>8s} {'tok/s':>8s} "
      f"{'correct':>7s} {'bound':>6s} {'b+lab':>6s} {'label':>6s} {'spur':>6s} {'missed':>6s}")
for n, v in STAGE1.items():
    e, ti = v["errors"], v["technique_info"]
    print(f"{n:22s} {v['macro_F1']:6.4f} {v['macro_P']:6.4f} {v['macro_R']:6.4f} {v['best_epoch']:3d} {ti['params_trainable']:>12,} "
          f"{ti['trainable_pct_of_backbone']:>8} {str(v['peak_mem_gb']):>6s} {v['train_seconds']:>8.0f} {v['tokens_per_sec']:>8.0f} "
          f"{e.get('correct',0):>7d} {e.get('boundary_error',0):>6d} {e.get('boundary_and_label_error',0):>6d} {e.get('label_error',0):>6d} "
          f"{e.get('spurious',0):>6d} {e.get('missed_entirely',0):>6d}")
print("\nPer-label F1 (fold1->fold2):")
print(f"{'setup':22s} " + " ".join(f"{l[:9]:>9s}" for l in LABELS))
for n, v in STAGE1.items():
    print(f"{n:22s} " + " ".join(f"{v['per_label_F1'][l]:9.3f}" for l in LABELS))
if STAGE2:
    print("\nPer-label F1 (fold2->fold1, confirmed setups):")
    for n, v in STAGE2.items():
        print(f"{n:22s} " + " ".join(f"{v['per_label_F1'][l]:9.3f}" for l in LABELS))
print("\nRefits:", FINAL)
print("\nSUMMARY_JSON_START")
print(json.dumps(SUMMARY, default=str))
print("SUMMARY_JSON_END")
print("=" * 100)
print("Send me this notebook (with outputs) or", os.path.basename(sp))


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded p2s_mor2_summary.json to bucket kavincg23/cs781-models
SWEEP p2s_mor2 (mor, 120,842,946 params, sanity loss 5.217) -- 274.5 min
  1. lorma_pi_bilstm: fold2 0.3406, fold1 0.3536, mean 0.3471
  2. lorma_bilstm: fold2 0.3372, fold1 0.3399, mean 0.3386
  3. bitfit_bilstm: fold2 0.3308, fold1 0.3356, mean 0.3332
  4. lora_bilstm: fold2 0.3306, fold1 0.3337, mean 0.3322
  5. adapter_pfeiffer_bilstm: fold2 0.3287, fold1 0.3317, mean 0.3302
  6. adapter_bilstm: fold2 0.3244
  7. frozen_bilstm: fold2 0.3173
  8. partial_ft_top2_bilstm: fold2 0.3147
  9. full_ft_bilstm: fold2 0.2856
  10. lorma: fold2 0.2292
  11. adapter: fold2 0.2099
  12. full_ft: fold2 0.2061
Submission zips (top 4 ): ['submission_p2s_mor2_lorma_pi_bilstm.zip', 'submission_p2s_mor2_lorma_bilstm.zip', 'submission_p2s_mor2_bitfit_bilstm.zip', 'submission_p2s_mor2_lora_bilstm.zip']

Full table (fold1->fold2):
setup                      F1      P      R  ep    trainable        %     GB  train s    tok/s correct  bound  